# Doa yang Terbaik — Pipeline Lengkap (v5 + v7 → Super-Ensemble 10 Model + Stage-2 v8)

Notebook mandiri yang menghasilkan `submission_v8_super10.csv` dari nol:

| Bagian | Isi | Kapan dijalankan |
|---|---|---|
| 0 | Setup lingkungan & pemeriksaan output yang sudah ada | selalu |
| A | **Pipeline v5** (IndoBERT-large, XLM-R-large, IndoBERTweet, + backbone ekstra & varian seed) → `working/preds/` | hanya jika prediksi v5 belum lengkap |
| B | **Pipeline v7** (deobfuskasi 5-lapis, IndoBERT-large, XLM-R-large, IndoBERTweet, DeBERTa-v3-large, varian seed) → `working/preds_v7/` + `ensemble_config_v7.json` | hanya jika prediksi/konfigurasi v7 belum lengkap |
| C | **Super-Ensemble 10 model**: blend v7 (bobot ensemble v7, tanpa DeBERTa) + v5 (bobot tetap, tanpa E5), lalu **Stage-2 v8** (kalibrasi multi-regime per post + skor leksikal kontrastif) → `submission.csv` | selalu (CPU, beberapa menit) |

Kode bagian A dan B adalah sel asli notebook v5 dan v7 tanpa perubahan logika (hanya dibungkus kondisi), karena pra-pemrosesan, arsitektur, dan hyperparameter kedua versi berbeda. Model yang prediksinya sudah ada dimuat ulang otomatis per backbone, sehingga notebook dapat dilanjutkan bila sesi terputus.

**Data**: letakkan `train.csv`, `test.csv`, `kbbi_v6.1.0_full.csv`, `colloquial-indonesian-lexicon.csv` di `MyDrive/Action 2026/` (Colab) atau `./data/`. Estimasi waktu A100: v5 ±3,2 jam, v7 ±5,4 jam.

## 0. Setup lingkungan

In [ ]:
import os, sys, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'], check=False)

print('Colab:', IN_COLAB)
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout)
except FileNotFoundError:
    print('nvidia-smi tidak tersedia (tidak ada GPU).')

In [ ]:
import os, glob, json, gc

def resolve_work_dir():
    for d in ['/content/drive/MyDrive/Action 2026', '/content']:
        if os.path.isdir(d):
            return os.path.join(d, 'working')
    if os.path.isdir('/kaggle/working'):
        return '/kaggle/working'
    return os.path.abspath('./working')


WORK_DIR = resolve_work_dir()
os.makedirs(WORK_DIR, exist_ok=True)

# Model yang dipakai Super-Ensemble (bagian C). Bagian A/B dilewati bila semua prediksinya sudah ada.
V5_REQUIRED = ['indobert-large-p1', 'indobert-large-p1_seed142', 'xlm-roberta-large', 'xlm-roberta-large_seed142', 'indobertweet-base-uncased']
V7_REQUIRED = ['indobert-large-p1', 'indobert-large-p1_seed142', 'xlm-roberta-large', 'xlm-roberta-large_seed142', 'indobertweet-base-uncased']
V5_DIR, V7_DIR = os.path.join(WORK_DIR, 'preds'), os.path.join(WORK_DIR, 'preds_v7')
V7_CFG_PATH = os.path.join(WORK_DIR, 'ensemble_config_v7.json')
FORCE_V5 = False   # True = paksa jalankan bagian A walau prediksi sudah ada (model yang ada tetap dimuat ulang)
FORCE_V7 = False


def preds_complete(d, tags):
    return all(os.path.isfile(os.path.join(d, f'{p}_{t}.npy')) for t in tags for p in ('oof', 'test'))


def find_v7_config():
    # konfigurasi ensemble hasil notebook v7 (bobot per pilar); nama lama juga diterima
    cands = [V7_CFG_PATH, os.path.join(WORK_DIR, 'ensemble_config_v7_backup.json')]
    base = os.path.dirname(WORK_DIR)
    for depth in ('', '*', '*/*'):
        cands += sorted(glob.glob(os.path.join(base, depth, 'ensemble_config_v7*.json')))
    return next((p for p in cands if os.path.isfile(p)), None)


RUN_V5 = FORCE_V5 or not preds_complete(V5_DIR, V5_REQUIRED)
RUN_V7 = FORCE_V7 or not preds_complete(V7_DIR, V7_REQUIRED) or find_v7_config() is None
print('Work dir :', WORK_DIR)
print(f'v5 lengkap: {preds_complete(V5_DIR, V5_REQUIRED)} -> RUN_V5 = {RUN_V5}')
print(f'v7 lengkap: {preds_complete(V7_DIR, V7_REQUIRED)} | config v7: {find_v7_config()} -> RUN_V7 = {RUN_V7}')

## A. Pipeline v5 (kondisional)
Sel asli `Doa_yang_terbaik_Data_Mining_v5.ipynb` (bagian 0–7). Sel audit & submission v5 tidak disertakan agar tidak menimpa output akhir.

### A.3 Konfigurasi & perangkat (v5)

In [ ]:
if RUN_V5:
    import os, re, gc, sys, json, math, time, glob, html, random, shutil, warnings, unicodedata
    from collections import Counter
    
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
    from transformers.utils import logging as hf_logging
    
    warnings.filterwarnings('ignore')
    hf_logging.set_verbosity_error()
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    START_TIME = time.time()
    
    
    def resolve_work_dir():
        for d in ['/content/drive/MyDrive/Action 2026', '/content']:
            if os.path.isdir(d):
                return os.path.join(d, 'working')
        if os.path.isdir('/kaggle/working'):
            return '/kaggle/working'
        return os.path.abspath('./working')
    
    
    class CFG:
        seed = 42
        n_splits = 5
        max_len = 256                     # A100: target 0% truncation
        max_len_cap = 384                 # dinaikkan otomatis bila masih ada teks > 256 token
        batch_size = 64                   # batch efektif (micro-batch x grad accumulation)
        eval_batch_size = 128
        epochs = 4
        lr = 1.5e-5
        head_lr = 8e-5
        weight_decay = 0.01
        llrd = 0.90
        warmup_ratio = 0.08
        grad_clip = 1.0
        dropout_ps = [0.1, 0.2, 0.3, 0.4, 0.5]
        pos_weight_min = 1.0
        pos_weight_max = 2.5
        target_f1 = 0.93
        time_budget_hours = 22.0
        base_backbones = [
            'indobenchmark/indobert-large-p1',
            'xlm-roberta-large',
            'indolem/indobertweet-base-uncased',
        ]
        extra_backbones = [
            'cahya/indobert-large-indonesian-522M',
            'intfloat/multilingual-e5-large-instruct',
        ]
        seed_variants = ['indobenchmark/indobert-large-p1', 'xlm-roberta-large']
        n_seed_variants = 1
        num_workers = 4
        work_dir = resolve_work_dir()
    
    
    LABELS = ['Infrastruktur', 'Kesehatan', 'Administrasi_Perizinan', 'Lingkungan_Kebersihan',
              'Ketertiban_Umum', 'Transportasi_LaluLintas', 'Sosial_Bantuan', 'Keamanan_Kriminalitas', 'Pendidikan']
    N_LABELS = len(LABELS)
    
    os.makedirs(CFG.work_dir, exist_ok=True)
    CKPT_DIR = os.path.join(CFG.work_dir, 'models_checkpoint')
    os.makedirs(CKPT_DIR, exist_ok=True)
    BEST_CKPT_PATH = os.path.join(CKPT_DIR, 'best_model.pth')
    BEST_META_PATH = os.path.join(CKPT_DIR, 'best_model_meta.json')
    PRED_DIR = os.path.join(CFG.work_dir, 'preds')
    os.makedirs(PRED_DIR, exist_ok=True)
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision('high')
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    else:
        GPU_NAME, GPU_MEM_GB = 'cpu', 0.0
    
    USE_AMP = DEVICE.type == 'cuda'
    USE_BF16 = USE_AMP and torch.cuda.is_bf16_supported()
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    USE_SCALER = USE_AMP and not USE_BF16
    
    
    def seed_everything(seed):
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    
    def elapsed_hours():
        return (time.time() - START_TIME) / 3600.0
    
    
    seed_everything(CFG.seed)
    print(f'Device: {DEVICE} | GPU: {GPU_NAME} ({GPU_MEM_GB:.0f} GB) | bf16: {USE_BF16} | '
          f'TF32: {DEVICE.type == "cuda"} | GradScaler: {USE_SCALER} | torch {torch.__version__}')
    if DEVICE.type == 'cuda' and 'A100' not in GPU_NAME:
        print('WARNING: GPU bukan A100 - backbone Large mungkin lambat / OOM (grad accumulation otomatis akan dipakai).')
    print('Work dir:', CFG.work_dir)
else:
    print("[lewati] RUN_V5 = False")


### A.5 Load data (v5)

In [ ]:
if RUN_V5:
    SEARCH_ROOTS = [
        ('/content/drive/MyDrive/Action 2026', True),
        ('/content/data', True),
        ('/content', False),
        ('/kaggle/input', True),
        ('./data', True),
        ('../data', True),
        ('.', False),
        ('..', False),
    ]
    
    
    def find_file(name, required=True):
        for root, recursive in SEARCH_ROOTS:
            if not os.path.isdir(root):
                continue
            direct = os.path.join(root, name)
            if os.path.isfile(direct):
                return os.path.abspath(direct)
            if recursive:
                hits = glob.glob(os.path.join(root, '**', name), recursive=True)
                if hits:
                    return os.path.abspath(sorted(hits, key=len)[0])
        if required:
            raise FileNotFoundError(f'{name} tidak ditemukan di {[r for r, _ in SEARCH_ROOTS]}')
        return None
    
    
    TRAIN_PATH = find_file('train.csv')
    TEST_PATH = find_file('test.csv')
    KBBI_PATH = find_file('kbbi_v6.1.0_full.csv', required=False)
    COLLOQ_PATH = find_file('colloquial-indonesian-lexicon.csv', required=False)
    for k, v in [('train', TRAIN_PATH), ('test', TEST_PATH), ('kbbi', KBBI_PATH), ('colloquial', COLLOQ_PATH)]:
        print(f'{k:>10}: {v}')
    
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    
    
    def pick_col(df, candidates, required=True):
        lower = {c.lower(): c for c in df.columns}
        for c in candidates:
            if c.lower() in lower:
                return lower[c.lower()]
        if required:
            raise KeyError(f'Kolom {candidates} tidak ada. Kolom tersedia: {list(df.columns)}')
        return None
    
    
    TEXT_CANDS = ['Teks', 'text', 'komentar', 'comment', 'content', 'tweet', 'teks_komentar']
    CAP_CANDS = ['Judul_Caption', 'caption', 'judul', 'title', 'post_caption', 'judul_post']
    ID_COL = pick_col(test_df, ['ID', 'id'])
    TR_TEXT, TE_TEXT = pick_col(train_df, TEXT_CANDS), pick_col(test_df, TEXT_CANDS)
    TR_CAP, TE_CAP = pick_col(train_df, CAP_CANDS, False), pick_col(test_df, CAP_CANDS, False)
    print('ID:', ID_COL, '| text:', TR_TEXT, '/', TE_TEXT, '| caption:', TR_CAP, '/', TE_CAP)
    
    missing = [l for l in LABELS if l not in train_df.columns]
    assert not missing, f'Label tidak ditemukan: {missing}'
    
    POS_TOKENS = {'1', '1.0', 'relevan', 'ya', 'yes', 'true', 'y'}
    
    
    def to_binary(s):
        if pd.api.types.is_numeric_dtype(s):
            return s.fillna(0).astype(float).round().clip(0, 1).astype(int)
        return s.astype(str).str.strip().str.lower().isin(POS_TOKENS).astype(int)
    
    
    for l in LABELS:
        train_df[l] = to_binary(train_df[l])
    Y = train_df[LABELS].values.astype(np.int64)
    N_TRAIN, N_TEST = len(train_df), len(test_df)
    
    print(f'Train rows: {N_TRAIN} (expected 23297) | Test rows: {N_TEST} (expected 34687)')
    if N_TRAIN != 23297:
        print('WARNING: jumlah baris train berbeda dari 23.297 - pastikan tidak ada baris yang dibuang.')
    print(pd.DataFrame({'prevalence_%': (Y.mean(0) * 100).round(2), 'positives': Y.sum(0)}, index=LABELS))
    print('Cardinality-0 rows:', int((Y.sum(1) == 0).sum()))
else:
    print("[lewati] RUN_V5 = False")


### A.7 Deobfuskasi (v5)

In [ ]:
if RUN_V5:
    URL_RE = re.compile(r'https?://\S+|www\.\S+')
    MENTION_RE = re.compile(r'@\w+')
    WS_RE = re.compile(r'\s+')
    TOKEN_RE = re.compile(r'[a-z0-9]+')
    
    
    def basic_clean(s):
        if not isinstance(s, str):
            return ''
        s = html.unescape(s)
        s = unicodedata.normalize('NFKC', s).lower()
        s = URL_RE.sub(' httpurl ', s)
        s = MENTION_RE.sub(' @user ', s)
        return WS_RE.sub(' ', s).strip()
    
    
    def load_wordlist_csv(path, preferred=('lemma', 'lema', 'kata', 'word', 'entri', 'entry', 'slang', 'formal')):
        words = set()
        if path is None or not os.path.isfile(path):
            return words
        df = pd.read_csv(path, dtype=str, on_bad_lines='skip', encoding_errors='ignore')
        cols = [c for c in df.columns if str(c).strip().lower() in preferred]
        if not cols:
            best, best_score = None, -1.0
            for c in df.columns:
                v = df[c].dropna().astype(str).head(3000)
                if len(v) == 0:
                    continue
                score = v.str.fullmatch(r'[A-Za-z][A-Za-z -]{0,30}').mean()
                if score > best_score:
                    best, best_score = c, score
            cols = [best] if best is not None else []
        for c in cols:
            for val in df[c].dropna().astype(str):
                toks = re.findall(r'[a-z]+', val.lower())
                words.update(t for t in toks if len(t) >= 2)
                joined = ''.join(toks)
                if 2 <= len(joined) <= 30:
                    words.add(joined)
        print(f'  {os.path.basename(path)}: kolom {cols} -> {len(words):,} kata')
        return words
    
    
    SEED_LEXICON = {
        'ktp', 'kk', 'sim', 'stnk', 'bpkb', 'npwp', 'skck', 'pdam', 'pln', 'bpjs', 'kis', 'kip', 'pkh', 'bansos', 'blt',
        'pungli', 'ppdb', 'sekolah', 'smkn', 'sman', 'smpn', 'sdn', 'puskesmas', 'rsud', 'satpol', 'dishub', 'polres',
        'polsek', 'polisi', 'damkar', 'tps', 'tpa', 'dukcapil', 'kelurahan', 'kecamatan', 'pemda', 'pemkot', 'pemkab',
    }
    
    SUFFIXES = ('nya', 'lah', 'kah', 'pun', 'ku', 'mu', 'kan', 'an', 'i')
    PREFIXES = (('meng', ''), ('meng', 'k'), ('meny', 's'), ('mem', ''), ('mem', 'p'), ('men', ''), ('men', 't'),
                ('me', ''), ('peng', ''), ('peng', 'k'), ('peny', 's'), ('pem', ''), ('pem', 'p'), ('pen', ''),
                ('pen', 't'), ('per', ''), ('pe', ''), ('ber', ''), ('be', ''), ('ter', ''), ('di', ''), ('ke', ''),
                ('se', ''))
    LEET = str.maketrans({'0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's', '7': 't', '8': 'b', '9': 'g'})
    LEET_ALT = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '7': 't', '8': 'b', '9': 'g'})
    
    
    def morph_stems(w):
        stems = {w}
        for _ in range(2):
            for st in list(stems):
                for s in SUFFIXES:
                    if st.endswith(s) and len(st) - len(s) >= 3:
                        stems.add(st[:-len(s)])
        for st in list(stems):
            for p, r in PREFIXES:
                if st.startswith(p) and len(st) - len(p) >= 2:
                    cand = r + st[len(p):]
                    if len(cand) >= 3:
                        stems.add(cand)
        stems.discard(w)
        return stems
    
    
    class CaesarDecoder:
        def __init__(self, lexicon, corpus_counter, min_len=4, plain_min_count=30, frequent_min_count=20):
            self.lex = lexicon
            self.cnt = corpus_counter
            self.min_len = min_len
            self.plain_min_count = plain_min_count
            self.frequent = {w for w, c in corpus_counter.items() if c >= frequent_min_count and w.isalpha()}
            letters = 'abcdefghijklmnopqrstuvwxyz'
            self.tables = [(k, str.maketrans(letters, letters[k:] + letters[:k])) for k in range(1, 26)]
            self.cache = {}
            self.ls_cache = {}
            self.n_changed = 0
    
        def lexical_strength(self, w):
            if w in self.ls_cache:
                return self.ls_cache[w]
            s = 0
            if w in self.lex:
                s = 3
            elif len(w) >= 6 and any(st in self.lex or st in self.frequent for st in morph_stems(w)):
                s = 2
            self.ls_cache[w] = s
            return s
    
        def target_strength(self, w):
            s = self.lexical_strength(w)
            if s == 0 and w in self.frequent:
                return 1
            return s
    
        def decode_word(self, w):
            if w in self.cache:
                return self.cache[w]
            res = w
            if len(w) >= self.min_len and w.isalpha() and self.lexical_strength(w) == 0:
                best, best_score = None, 0.0
                cnt_w = self.cnt.get(w, 0)
                for _, tab in self.tables:
                    c = w.translate(tab)
                    s = self.target_strength(c)
                    if s == 0:
                        continue
                    cnt_c = self.cnt.get(c, 0)
                    if len(w) <= 4 and (s < 3 or cnt_c < 3):
                        continue
                    if s == 1 and cnt_c < 5 * max(cnt_w, 1):
                        continue
                    score = s * 10.0 + math.log1p(cnt_c)
                    if score > best_score:
                        best, best_score = c, score
                if best is not None:
                    frequent_plain = cnt_w >= self.plain_min_count and cnt_w >= self.cnt.get(best, 0)
                    if not frequent_plain:
                        res = best
            self.cache[w] = res
            return res
    
        def decode_token(self, tok):
            if tok.isdigit():
                return tok
            if tok.isalpha():
                return self.decode_word(tok)
            for table in (LEET, LEET_ALT):
                t2 = tok.translate(table)
                if t2.isalpha() and len(t2) >= 3:
                    d = self.decode_word(t2)
                    if self.lexical_strength(d) >= 2:
                        return d
            return tok
    
        def decode_text(self, s):
            def repl(m):
                tok = m.group(0)
                out = self.decode_token(tok)
                if out != tok:
                    self.n_changed += 1
                return out
            return TOKEN_RE.sub(repl, s)
    
    
    print('Membangun leksikon ...')
    LEXICON = set(SEED_LEXICON)
    LEXICON |= load_wordlist_csv(KBBI_PATH)
    LEXICON |= load_wordlist_csv(COLLOQ_PATH)
    if KBBI_PATH is None:
        print('WARNING: kbbi_v6.1.0_full.csv tidak ditemukan - dekoder hanya memakai leksikon kolokial + korpus.')
    print(f'Total leksikon: {len(LEXICON):,} kata')
    
    for df, tcol, ccol in [(train_df, TR_TEXT, TR_CAP), (test_df, TE_TEXT, TE_CAP)]:
        df['text_clean'] = df[tcol].map(basic_clean)
        df['cap_clean'] = df[ccol].map(basic_clean) if ccol is not None else ''
    
    corpus_counter = Counter()
    for s in pd.concat([train_df['text_clean'], test_df['text_clean']]):
        corpus_counter.update(TOKEN_RE.findall(s))
    for s in pd.concat([train_df['cap_clean'], test_df['cap_clean']]).drop_duplicates():
        corpus_counter.update(TOKEN_RE.findall(s))
    print(f'Vocab korpus: {len(corpus_counter):,} token unik')
    
    decoder = CaesarDecoder(LEXICON, corpus_counter)
else:
    print("[lewati] RUN_V5 = False")


### A.8 Jalankan dekode (v5)

In [ ]:
if RUN_V5:
    t0 = time.time()
    cap_cache = {}
    
    
    def decode_caption(c):
        if c not in cap_cache:
            cap_cache[c] = decoder.decode_text(c)
        return cap_cache[c]
    
    
    for df in (train_df, test_df):
        df['text_dec'] = df['text_clean'].map(decoder.decode_text)
        df['cap_dec'] = df['cap_clean'].map(decode_caption)
    print(f'Dekode selesai dalam {time.time() - t0:.1f}s | token dipulihkan: {decoder.n_changed:,}')
    
    CHECKS = {'wlshfhuhu': 'pelayanan', 'fvzaln': 'simnya', 'kvtpzpsp': 'domisili', 'alwlv': 'palak',
              'ufxtpfs': 'pasokan', 'xmujmzqbipciv': 'pemberitahuan', 'zrarzonx': 'menembak'}
    n_ok = 0
    for enc_word, expected in CHECKS.items():
        got = decoder.decode_word(enc_word)
        ok = got == expected
        n_ok += int(ok)
        status = 'OK' if ok else 'MISS'
        print(f'  {enc_word:>15} -> {got:<15} (expected {expected:<14}) {status}')
    print(f'Verifikasi kata kunci: {n_ok}/{len(CHECKS)} terpulihkan')
    
    changed = train_df.loc[train_df['text_clean'] != train_df['text_dec'], ['text_clean', 'text_dec']].head(8)
    print(changed.to_string())
else:
    print("[lewati] RUN_V5 = False")


### A.10 Fold & pos_weight (v5)

In [ ]:
if RUN_V5:
    group_ids = np.zeros(N_TRAIN, dtype=np.int64)
    cap_to_gid = {}
    next_gid = 0
    n_twitter = 0
    for i, c in enumerate(train_df['cap_clean'].values):
        if isinstance(c, str) and c.strip():
            if c not in cap_to_gid:
                cap_to_gid[c] = next_gid
                next_gid += 1
            group_ids[i] = cap_to_gid[c]
        else:
            group_ids[i] = -1
            n_twitter += 1
    for i in np.where(group_ids == -1)[0]:
        group_ids[i] = next_gid
        next_gid += 1
    train_df['post_id'] = group_ids
    print(f'Postingan unik ber-caption: {len(cap_to_gid)} | baris tanpa caption (grup unik): {n_twitter}')
    
    prev = Y.mean(0)
    strat = np.full(N_TRAIN, N_LABELS, dtype=np.int64)
    for j in np.argsort(prev)[::-1]:
        strat[Y[:, j] == 1] = j
    
    folds = np.full(N_TRAIN, -1, dtype=np.int64)
    sgkf = StratifiedGroupKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
    for f, (_, va_idx) in enumerate(sgkf.split(np.zeros(N_TRAIN), strat, groups=group_ids)):
        folds[va_idx] = f
    train_df['fold'] = folds
    assert (folds >= 0).all()
    assert train_df.groupby('post_id')['fold'].nunique().max() == 1, 'Kebocoran grup antar fold!'
    print('Leak-free check OK (setiap post_id hanya berada di 1 fold).')
    print('Rows per fold:', np.bincount(folds).tolist())
    print(pd.DataFrame({f'fold{f}': Y[folds == f].mean(0).round(4) for f in range(CFG.n_splits)}, index=LABELS))
    
    n_pos = Y.sum(0).astype(np.float64)
    POS_WEIGHT = np.clip(np.sqrt((N_TRAIN - n_pos) / np.maximum(n_pos, 1.0)), CFG.pos_weight_min, CFG.pos_weight_max)
    POS_WEIGHT_T = torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)
    print(pd.Series(POS_WEIGHT.round(3), index=LABELS, name='pos_weight'))
else:
    print("[lewati] RUN_V5 = False")


### A.12 Tokenisasi, model, metrik (v5)

In [ ]:
if RUN_V5:
    def encode(tok, texts, caps, max_len, truncate=True):
        res = [None] * len(texts)
        pair_idx = [i for i, c in enumerate(caps) if c]
        single_idx = [i for i, c in enumerate(caps) if not c]
        kw = dict(truncation='longest_first', max_length=max_len) if truncate else dict(truncation=False)
        if pair_idx:
            e = tok([texts[i] for i in pair_idx], [caps[i] for i in pair_idx], **kw)
            keys = list(e.keys())
            for k, i in enumerate(pair_idx):
                res[i] = {key: e[key][k] for key in keys}
        if single_idx:
            e = tok([texts[i] for i in single_idx], **kw)
            keys = list(e.keys())
            for k, i in enumerate(single_idx):
                res[i] = {key: e[key][k] for key in keys}
        keys = list(res[0].keys())
        return {key: [r[key] for r in res] for key in keys}
    
    
    class PairDataset(Dataset):
        def __init__(self, enc, indices, labels=None):
            self.enc, self.indices, self.labels = enc, np.asarray(indices), labels
    
        def __len__(self):
            return len(self.indices)
    
        def __getitem__(self, i):
            idx = self.indices[i]
            item = {k: v[idx] for k, v in self.enc.items()}
            if self.labels is not None:
                item['labels'] = self.labels[idx]
            return item
    
    
    def make_collate(tok):
        def collate(batch):
            labels = None
            if 'labels' in batch[0]:
                labels = torch.tensor(np.stack([b.pop('labels') for b in batch]), dtype=torch.float32)
            out = dict(tok.pad(batch, padding='longest', pad_to_multiple_of=8, return_tensors='pt'))
            if labels is not None:
                out['labels'] = labels
            return out
        return collate
    
    
    def make_loader(enc, indices, tok, labels=None, shuffle=False, sort_by_len=False, batch_size=32):
        indices = np.asarray(indices)
        if sort_by_len:
            lens = np.array([len(enc['input_ids'][i]) for i in indices])
            indices = indices[np.argsort(lens, kind='stable')]
        ds = PairDataset(enc, indices, labels)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=make_collate(tok),
                            num_workers=CFG.num_workers, pin_memory=USE_AMP, drop_last=False)
        return loader, indices
    
    
    class MultiLabelModel(nn.Module):
        def __init__(self, name, n_labels, dropout_ps, prior=None):
            super().__init__()
            config = AutoConfig.from_pretrained(name)
            self.backbone = AutoModel.from_pretrained(name, config=config)
            hidden = config.hidden_size
            self.feat_dim = 2 * hidden  # 2.048 untuk model Large (hidden 1024), 1.536 untuk Base
            self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropout_ps])
            self.head = nn.Linear(self.feat_dim, n_labels)
            nn.init.normal_(self.head.weight, std=0.02)
            if prior is not None:
                p = np.clip(np.asarray(prior, dtype=np.float64), 1e-4, 1 - 1e-4)
                self.head.bias.data = torch.tensor(np.log(p / (1 - p)), dtype=torch.float32)
            else:
                nn.init.zeros_(self.head.bias)
    
        def forward(self, input_ids, attention_mask, token_type_ids=None):
            kw = dict(input_ids=input_ids, attention_mask=attention_mask)
            if token_type_ids is not None:
                kw['token_type_ids'] = token_type_ids
            hs = self.backbone(**kw).last_hidden_state.float()
            cls = hs[:, 0]
            m = attention_mask.unsqueeze(-1).to(hs.dtype)
            mean = (hs * m).sum(1) / m.sum(1).clamp(min=1e-6)
            feat = torch.cat([cls, mean], dim=-1)
            logits = torch.stack([self.head(d(feat)) for d in self.dropouts], dim=0).mean(0)
            return logits
    
    
    NO_DECAY = ('bias', 'LayerNorm.weight', 'LayerNorm.bias', 'layer_norm', 'layernorm')
    
    
    def get_param_groups(model):
        groups = []
    
        def add(named, lr):
            dec = [p for n, p in named if p.requires_grad and not any(nd in n for nd in NO_DECAY)]
            nod = [p for n, p in named if p.requires_grad and any(nd in n for nd in NO_DECAY)]
            if dec:
                groups.append({'params': dec, 'lr': lr, 'weight_decay': CFG.weight_decay})
            if nod:
                groups.append({'params': nod, 'lr': lr, 'weight_decay': 0.0})
    
        bb = model.backbone
        enc = getattr(bb, 'encoder', None)
        layers = list(enc.layer) if enc is not None and hasattr(enc, 'layer') else []
        assigned = set()
        for depth, layer in enumerate(reversed(layers)):
            named = list(layer.named_parameters())
            add(named, CFG.lr * (CFG.llrd ** depth))
            assigned |= {id(p) for _, p in named}
        rest = [(n, p) for n, p in bb.named_parameters() if id(p) not in assigned]
        add(rest, CFG.lr * (CFG.llrd ** len(layers)))
        add(list(model.head.named_parameters()), CFG.head_lr)
        return groups
    
    
    @torch.no_grad()
    def predict(model, loader):
        model.eval()
        preds = []
        for b in loader:
            b.pop('labels', None)
            b = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(**b)
            preds.append(torch.sigmoid(logits.float()).cpu().numpy())
        return np.concatenate(preds, 0)
    
    
    THR_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
    
    
    def macro_f1_grid(y, p, thr=THR_GRID):
        y = np.asarray(y).astype(np.int64)
        p = np.asarray(p, dtype=np.float64)
        thr = np.asarray(thr, dtype=np.float64)
        order = np.argsort(-p, kind='stable')
        ps, ys = p[order], y[order]
        n_pred = np.searchsorted(-ps, -thr, side='right')
        ctp = np.concatenate([[0], np.cumsum(ys)])
        tp = ctp[n_pred]
        fp = n_pred - tp
        n_pos_true = ys.sum()
        fn = n_pos_true - tp
        tn = len(ys) - n_pos_true - fp
        f1_pos = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        f1_neg = 2 * tn / np.maximum(2 * tn + fp + fn, 1)
        return (f1_pos + f1_neg) / 2.0
    
    
    def macro_f1_at(y, p, t):
        return float(macro_f1_grid(y, p, np.array([t]))[0])
    
    
    def per_pillar_best(y, P):
        rows = []
        for j in range(P.shape[1]):
            s = macro_f1_grid(y[:, j], P[:, j])
            k = int(np.argmax(s))
            rows.append((float(THR_GRID[k]), float(s[k])))
        return rows
    
    
    def maybe_save_global_best(state, backbone, fold, score, max_len):
        meta = {'score': -1.0}
        if os.path.isfile(BEST_META_PATH):
            with open(BEST_META_PATH) as fh:
                meta = json.load(fh)
        if score > meta.get('score', -1.0):
            torch.save(state, BEST_CKPT_PATH)
            meta = {'score': float(score), 'backbone': backbone, 'fold': int(fold), 'max_len': int(max_len),
                    'pooling': 'cls+mean', 'labels': LABELS}
            with open(BEST_META_PATH, 'w') as fh:
                json.dump(meta, fh, indent=2)
            print(f'  >> best_model.pth diperbarui ({backbone}, fold {fold}, score {score:.4f})')
        for p in glob.glob(os.path.join(CFG.work_dir, '**', '*.pth'), recursive=True):
            if os.path.abspath(p) != os.path.abspath(BEST_CKPT_PATH):
                os.remove(p)
else:
    print("[lewati] RUN_V5 = False")


### A.14 Training per backbone (v5)

In [ ]:
if RUN_V5:
    BACKBONE_HOURS = {}
    
    
    def model_tag(name, seed_offset=0):
        tag = name.split('/')[-1]
        return tag if seed_offset == 0 else f'{tag}_seed{CFG.seed + 100 * seed_offset}'
    
    
    def choose_max_len(name, tok, tr_texts, tr_caps, te_texts, te_caps):
        config = AutoConfig.from_pretrained(name)
        limit = int(getattr(config, 'max_position_embeddings', 512) or 512)
        if getattr(config, 'model_type', '') in ('roberta', 'xlm-roberta', 'camembert'):
            limit -= 2
        limit = min(limit, 512)
        lens = []
        for texts, caps in ((tr_texts, tr_caps), (te_texts, te_caps)):
            full = encode(tok, texts, caps, None, truncate=False)
            lens.extend(len(x) for x in full['input_ids'])
            del full
        lens = np.array(lens)
        ml = CFG.max_len
        if (lens > ml).any():
            need = int(math.ceil(lens.max() / 8.0) * 8)
            ml = max(CFG.max_len, min(need, CFG.max_len_cap))
        ml = min(ml, limit)
        print(f'  token median={np.median(lens):.0f} | p90={np.percentile(lens, 90):.0f} | '
              f'p99={np.percentile(lens, 99):.0f} | max={lens.max()} | terpotong@128={np.mean(lens > 128) * 100:.1f}% | '
              f'terpotong@256={np.mean(lens > 256) * 100:.2f}%')
        print(f'  max_len dipakai = {ml} -> baris terpotong = {np.mean(lens > ml) * 100:.2f}%')
        return ml
    
    
    def initial_accum(name, ml):
        big = any(k in name.lower() for k in ('large', '522m'))
        if not big or GPU_MEM_GB >= 70:
            return 1
        return 1 if ml <= 256 else 2
    
    
    def is_oom(e):
        return 'out of memory' in str(e).lower()
    
    
    def free_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    
    def train_one_fold(name, fold, seed, enc_tr, tok, ml, accum, te_loader, te_order):
        f_start = time.time()
        seed_everything(seed + fold)
        Yf = Y.astype(np.float32)
        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]
        micro_bs = max(1, CFG.batch_size // accum)
        tr_loader, _ = make_loader(enc_tr, tr_idx, tok, labels=Yf, shuffle=True, batch_size=micro_bs)
        va_loader, va_order = make_loader(enc_tr, va_idx, tok, sort_by_len=True, batch_size=CFG.eval_batch_size)
    
        model = MultiLabelModel(name, N_LABELS, CFG.dropout_ps, prior=Y[tr_idx].mean(0)).to(DEVICE)
        if fold == 0:
            n_params = sum(p.numel() for p in model.backbone.parameters()) / 1e6
            print(f'  params backbone={n_params:.0f}M | fitur head={model.feat_dim} dim | '
                  f'micro-batch={micro_bs} x accum={accum} = batch efektif {micro_bs * accum} | max_len={ml}')
        optimizer = torch.optim.AdamW(get_param_groups(model), lr=CFG.lr, eps=1e-6)
        n_batches = len(tr_loader)
        steps_per_epoch = math.ceil(n_batches / accum)
        total_steps = steps_per_epoch * CFG.epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT_T)
    
        best_score, best_state, best_val = -1.0, None, None
        for epoch in range(CFG.epochs):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            running, n_seen = 0.0, 0
            for step, batch in enumerate(tr_loader):
                labels = batch.pop('labels').to(DEVICE, non_blocking=True)
                batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    logits = model(**batch)
                loss = loss_fn(logits.float(), labels)
                scaler.scale(loss / accum).backward()
                if (step + 1) % accum == 0 or (step + 1) == n_batches:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
                running += loss.item() * labels.size(0)
                n_seen += labels.size(0)
    
            val_sorted = predict(model, va_loader)
            val = np.empty((len(va_idx), N_LABELS), dtype=np.float32)
            val[np.searchsorted(va_idx, va_order)] = val_sorted
            scores = per_pillar_best(Y[va_idx], val)
            score = float(np.mean([s for _, s in scores]))
            print(f'  fold {fold} ep {epoch + 1}/{CFG.epochs} | loss {running / max(n_seen, 1):.4f} | '
                  f'val mean macroF1 {score:.4f} | min pilar {min(s for _, s in scores):.4f} | '
                  f'{(time.time() - f_start) / 60:.1f} min')
            if score > best_score:
                best_score, best_val = score, val
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    
        model.load_state_dict(best_state)
        te_sorted = predict(model, te_loader)
        te = np.empty_like(te_sorted)
        te[te_order] = te_sorted
        del model, optimizer, scheduler, scaler, tr_loader, va_loader
        free_cuda()
        return best_score, best_val, best_state, te
    
    
    def train_backbone(name, seed_offset=0):
        tag = model_tag(name, seed_offset)
        seed = CFG.seed + 100 * seed_offset
        oof_path = os.path.join(PRED_DIR, f'oof_{tag}.npy')
        test_path = os.path.join(PRED_DIR, f'test_{tag}.npy')
        partial_path = os.path.join(PRED_DIR, f'partial_{tag}.npz')
        if os.path.isfile(oof_path) and os.path.isfile(test_path):
            print(f'[{tag}] prediksi sudah ada - dimuat ulang.')
            return np.load(oof_path), np.load(test_path)
    
        t_start = time.time()
        print('=' * 90)
        print(f'Backbone: {name} (seed {seed}) -> tag {tag}')
        tok = AutoTokenizer.from_pretrained(name)
        tr_texts, tr_caps = train_df['text_dec'].tolist(), train_df['cap_dec'].tolist()
        te_texts, te_caps = test_df['text_dec'].tolist(), test_df['cap_dec'].tolist()
    
        ml = choose_max_len(name, tok, tr_texts, tr_caps, te_texts, te_caps)
        enc_tr = encode(tok, tr_texts, tr_caps, ml)
        enc_te = encode(tok, te_texts, te_caps, ml)
        te_loader, te_order = make_loader(enc_te, np.arange(N_TEST), tok, sort_by_len=True, batch_size=CFG.eval_batch_size)
    
        oof = np.zeros((N_TRAIN, N_LABELS), dtype=np.float32)
        test_pred = np.zeros((N_TEST, N_LABELS), dtype=np.float32)
        done = []
        if os.path.isfile(partial_path):
            z = np.load(partial_path)
            oof, test_pred, done = z['oof'], z['test'], z['done'].tolist()
            print(f'  Resume dari fold yang sudah selesai: {done}')
    
        for fold in range(CFG.n_splits):
            if fold in done:
                continue
            accum = initial_accum(name, ml)
            while True:
                oom = False
                try:
                    score, val, state, te = train_one_fold(name, fold, seed, enc_tr, tok, ml, accum, te_loader, te_order)
                except RuntimeError as e:
                    if not is_oom(e):
                        raise
                    oom = True
                if not oom:
                    break
                free_cuda()
                accum *= 2
                if accum > 8:
                    raise RuntimeError(f'OOM persisten pada {name} bahkan dengan grad accumulation 8')
                print(f'  OOM -> grad accumulation dinaikkan ke {accum} (micro-batch {CFG.batch_size // accum})')
    
            va_idx = np.where(folds == fold)[0]
            oof[va_idx] = val
            test_pred += te / CFG.n_splits
            maybe_save_global_best(state, tag, fold, score, ml)
            del state
            free_cuda()
            done.append(fold)
            np.savez(partial_path, oof=oof, test=test_pred, done=np.array(done, dtype=np.int64))
    
        np.save(oof_path, oof)
        np.save(test_path, test_pred)
        if os.path.isfile(partial_path):
            os.remove(partial_path)
        hours = (time.time() - t_start) / 3600.0
        BACKBONE_HOURS[tag] = hours
        scores = per_pillar_best(Y, oof)
        print(f'[{tag}] OOF mean macroF1 (thr per pilar) = {np.mean([s for _, s in scores]):.4f} | {hours:.2f} jam')
        for l, (t, s) in zip(LABELS, scores):
            flag = 'PASS' if s >= CFG.target_f1 else 'FAIL'
            print(f'    {l:<26} thr={t:.2f}  macroF1={s:.4f}  {flag}')
        return oof, test_pred
else:
    print("[lewati] RUN_V5 = False")


### A.16 Stacking & optimasi ensemble (v5)

In [ ]:
if RUN_V5:
    def to_logit(p, eps=1e-6):
        p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
        return np.log(p / (1 - p))
    
    
    def stack_cross_pillar(oof_dict, test_dict, C=0.5):
        names = list(oof_dict.keys())
        Xtr = np.concatenate([to_logit(oof_dict[n]) for n in names], 1)
        Xte = np.concatenate([to_logit(test_dict[n]) for n in names], 1)
        oof_meta = np.zeros((N_TRAIN, N_LABELS), dtype=np.float64)
        test_meta = np.zeros((N_TEST, N_LABELS), dtype=np.float64)
        for j in range(N_LABELS):
            for f in range(CFG.n_splits):
                tr, va = folds != f, folds == f
                clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))
                clf.fit(Xtr[tr], Y[tr, j])
                oof_meta[va, j] = clf.predict_proba(Xtr[va])[:, 1]
                test_meta[:, j] += clf.predict_proba(Xte)[:, 1] / CFG.n_splits
        return oof_meta, test_meta
    
    
    def simplex_grid(m, step):
        n = int(round(1.0 / step))
        out = []
    
        def rec(prefix, remaining, k):
            if k == 1:
                out.append(prefix + [remaining])
                return
            for i in range(remaining + 1):
                rec(prefix + [i], remaining - i, k - 1)
    
        rec([], n, m)
        return np.array(out, dtype=np.float64) / n
    
    
    def default_weight_step(m):
        if m <= 3:
            return 0.05
        if m == 4:
            return 0.1
        if m <= 7:
            return 0.2
        return 0.25
    
    
    def optimize_pillar_ensemble(oof_dict, test_dict, y, weight_step=None, thr_grid=THR_GRID, verbose=True):
        names = list(oof_dict.keys())
        m = len(names)
        if weight_step is None:
            weight_step = default_weight_step(m)
        W = simplex_grid(m, weight_step)
        n_tr = len(y)
        n_te = len(next(iter(test_dict.values())))
        out = {'names': names, 'weights': np.zeros((N_LABELS, m)), 'thresholds': np.zeros(N_LABELS),
               'scores': np.zeros(N_LABELS), 'oof_prob': np.zeros((n_tr, N_LABELS)),
               'test_prob': np.zeros((n_te, N_LABELS))}
        for j in range(N_LABELS):
            O = np.stack([np.asarray(oof_dict[n][:, j], dtype=np.float64) for n in names], 0)
            T = np.stack([np.asarray(test_dict[n][:, j], dtype=np.float64) for n in names], 0)
            best_s, best_w, best_t = -1.0, W[0], 0.5
            for w in W:
                s = macro_f1_grid(y[:, j], w @ O, thr_grid)
                k = int(np.argmax(s))
                if s[k] > best_s + 1e-9:
                    best_s, best_w, best_t = float(s[k]), w, float(thr_grid[k])
            out['weights'][j] = best_w
            out['thresholds'][j] = best_t
            out['scores'][j] = best_s
            out['oof_prob'][:, j] = best_w @ O
            out['test_prob'][:, j] = best_w @ T
        out['oof_bin'] = (out['oof_prob'] >= out['thresholds'][None, :]).astype(int)
        out['test_bin'] = (out['test_prob'] >= out['thresholds'][None, :]).astype(int)
        if verbose:
            df = pd.DataFrame(out['weights'].round(2), index=LABELS, columns=names)
            df['thr'] = out['thresholds']
            df['macroF1'] = out['scores'].round(5)
            print(df.to_string())
        return out
    
    
    def run_ensemble(oof_dict, test_dict):
        oof_all, test_all = dict(oof_dict), dict(test_dict)
        sm_oof, sm_test = stack_cross_pillar(oof_dict, test_dict)
        oof_all['stack_lr'], test_all['stack_lr'] = sm_oof, sm_test
        st = per_pillar_best(Y, sm_oof)
        print(f'Stacking lintas-pilar OOF mean macroF1 = {np.mean([s for _, s in st]):.5f}')
        res = optimize_pillar_ensemble(oof_all, test_all, Y)
        return res, oof_all, test_all
    
    
    def honest_cv_scores(oof_dict, y):
        names = list(oof_dict.keys())
        pred = np.zeros_like(y)
        for f in range(CFG.n_splits):
            tr, va = folds != f, folds == f
            r = optimize_pillar_ensemble({n: oof_dict[n][tr] for n in names},
                                         {n: oof_dict[n][va] for n in names}, y[tr], verbose=False)
            pred[va] = r['test_bin']
        return np.array([macro_f1_at(y[:, j], pred[:, j].astype(float), 0.5) for j in range(N_LABELS)])
    
    
    def audit(res, honest=None):
        rows = []
        for j, l in enumerate(LABELS):
            r = {'pillar': l, 'oof_macroF1': round(float(res['scores'][j]), 5), 'thr': float(res['thresholds'][j])}
            if honest is not None:
                r['nested_cv_macroF1'] = round(float(honest[j]), 5)
            r['status'] = 'PASS' if res['scores'][j] >= CFG.target_f1 else 'FAIL'
            rows.append(r)
        print(pd.DataFrame(rows).to_string(index=False))
        mean_f1 = float(np.mean(res['scores']))
        n_pass = int((res['scores'] >= CFG.target_f1).sum())
        print(f'Mean macro F1 (OOF) = {mean_f1:.5f} | pilar >= {CFG.target_f1}: {n_pass}/{N_LABELS}')
        if honest is not None:
            print(f'Mean macro F1 (nested CV, estimasi jujur) = {float(np.mean(honest)):.5f}')
        return n_pass == N_LABELS
else:
    print("[lewati] RUN_V5 = False")


### A.18 Eksekusi training + iterasi otomatis (v5)

In [ ]:
if RUN_V5:
    OOF, TEST = {}, {}
    for name in CFG.base_backbones:
        tag = model_tag(name)
        OOF[tag], TEST[tag] = train_backbone(name)
    
    RESULT, OOF_ALL, TEST_ALL = run_ensemble(OOF, TEST)
    ALL_PASS = audit(RESULT)
    
    queue = [(b, 0) for b in CFG.extra_backbones]
    queue += [(b, k) for k in range(1, CFG.n_seed_variants + 1) for b in CFG.seed_variants]
    queue = [(b, s) for b, s in queue if model_tag(b, s) not in OOF]
    
    while not ALL_PASS and queue:
        est = max(BACKBONE_HOURS.values()) * 1.15 if BACKBONE_HOURS else 1.5
        if elapsed_hours() + est > CFG.time_budget_hours:
            print(f'Sisa waktu tidak cukup (terpakai {elapsed_hours():.2f} jam, estimasi {est:.2f} jam/backbone).')
            break
        name, s_off = queue.pop(0)
        tag = model_tag(name, s_off)
        failing = [l for j, l in enumerate(LABELS) if RESULT['scores'][j] < CFG.target_f1]
        print(f'Pilar < {CFG.target_f1}: {failing} -> menambah backbone: {tag}')
        try:
            OOF[tag], TEST[tag] = train_backbone(name, s_off)
        except Exception as e:
            print(f'  Gagal melatih {tag}: {e}')
            free_cuda()
            continue
        RESULT, OOF_ALL, TEST_ALL = run_ensemble(OOF, TEST)
        ALL_PASS = audit(RESULT)
    
    print(f'Total waktu: {elapsed_hours():.2f} jam | backbone dipakai: {list(OOF.keys())}')
else:
    print("[lewati] RUN_V5 = False")


In [ ]:
if RUN_V5:
    for _n in ['OOF', 'TEST', 'RESULT', 'OOF_ALL', 'TEST_ALL']:
        globals().pop(_n, None)
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
print('v5 lengkap:', preds_complete(V5_DIR, V5_REQUIRED))

## B. Pipeline v7 (kondisional)
Sel asli `Doa_yang_terbaik_Data_Mining_v7.ipynb` (bagian 0–8). Konfigurasi ensemble v7 disimpan sebagai `ensemble_config_v7.json` (dibutuhkan bagian C). Sel submission v7 tidak disertakan.

### B.3 Konfigurasi & perangkat (v7)

In [ ]:
if RUN_V7:
    import os, re, gc, sys, json, math, time, glob, html, random, shutil, warnings, unicodedata
    from collections import Counter, defaultdict
    
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from sklearn.linear_model import LogisticRegression, RidgeClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import make_pipeline, FeatureUnion
    from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
    from transformers.utils import logging as hf_logging
    
    warnings.filterwarnings('ignore')
    hf_logging.set_verbosity_error()
    os.environ['TOKENIZERS_PARALLELISM'] = 'true'
    START_TIME = time.time()
    
    
    def resolve_work_dir():
        for d in ['/content/drive/MyDrive/Action 2026', '/content']:
            if os.path.isdir(d):
                return os.path.join(d, 'working')
        if os.path.isdir('/kaggle/working'):
            return '/kaggle/working'
        return os.path.abspath('./working')
    
    
    class CFG:
        seed = 42
        n_splits = 5
        max_len = 256
        max_len_cap = 384
        batch_size = 64
        eval_batch_size = 128
        epochs = 4
        lr = 1.5e-5
        head_lr = 8e-5
        weight_decay = 0.01
        llrd = 0.90
        warmup_ratio = 0.08
        grad_clip = 1.0
        dropout_ps = [0.1, 0.2, 0.3, 0.4, 0.5]
        pos_weight_min = 1.0
        pos_weight_max = 3.0
        target_f1 = 0.95
        time_budget_hours = 22.0
        base_backbones = [
            'indobenchmark/indobert-large-p1',
            'xlm-roberta-large',
            'indolem/indobertweet-base-uncased',
        ]
        extra_backbones = [
            'microsoft/deberta-v3-large',
        ]
        seed_variants = ['indobenchmark/indobert-large-p1', 'xlm-roberta-large']
        n_seed_variants = 1
        num_workers = 0  # 0 mencegah deadlock os.fork() dengan tokenizers/OpenMP di Linux/Colab
        work_dir = resolve_work_dir()
    
    
    LABELS = ['Infrastruktur', 'Kesehatan', 'Administrasi_Perizinan', 'Lingkungan_Kebersihan',
              'Ketertiban_Umum', 'Transportasi_LaluLintas', 'Sosial_Bantuan', 'Keamanan_Kriminalitas', 'Pendidikan']
    N_LABELS = len(LABELS)
    
    os.makedirs(CFG.work_dir, exist_ok=True)
    CKPT_DIR = os.path.join(CFG.work_dir, 'models_checkpoint')
    os.makedirs(CKPT_DIR, exist_ok=True)
    BEST_CKPT_PATH = os.path.join(CKPT_DIR, 'best_model.pth')
    BEST_META_PATH = os.path.join(CKPT_DIR, 'best_model_meta.json')
    PRED_DIR = os.path.join(CFG.work_dir, 'preds_v7')
    os.makedirs(PRED_DIR, exist_ok=True)
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if DEVICE.type == 'cuda':
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision('high')
        GPU_NAME = torch.cuda.get_device_name(0)
        GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    else:
        GPU_NAME, GPU_MEM_GB = 'cpu', 0.0
    
    USE_AMP = DEVICE.type == 'cuda'
    USE_BF16 = USE_AMP and torch.cuda.is_bf16_supported()
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    USE_SCALER = USE_AMP and not USE_BF16
    
    
    def seed_everything(seed):
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    
    def elapsed_hours():
        return (time.time() - START_TIME) / 3600.0
    
    
    seed_everything(CFG.seed)
    print(f'Device: {DEVICE} | GPU: {GPU_NAME} ({GPU_MEM_GB:.0f} GB) | bf16: {USE_BF16} | '
          f'TF32: {DEVICE.type == "cuda"} | GradScaler: {USE_SCALER} | torch {torch.__version__}')
    print('Work dir:', CFG.work_dir, '| Pred dir:', PRED_DIR)
else:
    print("[lewati] RUN_V7 = False")


### B.5 Load data (v7)

In [ ]:
if RUN_V7:
    SEARCH_ROOTS = [
        ('/content/drive/MyDrive/Action 2026', True),
        ('/content/data', True),
        ('/content', False),
        ('/kaggle/input', True),
        ('./data', True),
        ('../data', True),
        ('.', False),
        ('..', False),
    ]
    
    
    def find_file(name, required=True):
        for root, recursive in SEARCH_ROOTS:
            if not os.path.isdir(root):
                continue
            direct = os.path.join(root, name)
            if os.path.isfile(direct):
                return os.path.abspath(direct)
            if recursive:
                hits = glob.glob(os.path.join(root, '**', name), recursive=True)
                if hits:
                    return os.path.abspath(sorted(hits, key=len)[0])
        if required:
            raise FileNotFoundError(f'{name} tidak ditemukan di {[r for r, _ in SEARCH_ROOTS]}')
        return None
    
    
    TRAIN_PATH = find_file('train.csv')
    TEST_PATH = find_file('test.csv')
    KBBI_PATH = find_file('kbbi_v6.1.0_full.csv', required=False)
    COLLOQ_PATH = find_file('colloquial-indonesian-lexicon.csv', required=False)
    for k, v in [('train', TRAIN_PATH), ('test', TEST_PATH), ('kbbi', KBBI_PATH), ('colloquial', COLLOQ_PATH)]:
        print(f'{k:>10}: {v}')
    
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    
    
    def pick_col(df, candidates, required=True):
        lower = {c.lower(): c for c in df.columns}
        for c in candidates:
            if c.lower() in lower:
                return lower[c.lower()]
        if required:
            raise KeyError(f'Kolom {candidates} tidak ada. Kolom tersedia: {list(df.columns)}')
        return None
    
    
    TEXT_CANDS = ['Teks', 'text', 'komentar', 'comment', 'content', 'tweet', 'teks_komentar']
    CAP_CANDS = ['Judul_Caption', 'caption', 'judul', 'title', 'post_caption', 'judul_post']
    ID_COL = pick_col(test_df, ['ID', 'id'])
    TR_TEXT, TE_TEXT = pick_col(train_df, TEXT_CANDS), pick_col(test_df, TEXT_CANDS)
    TR_CAP, TE_CAP = pick_col(train_df, CAP_CANDS, False), pick_col(test_df, CAP_CANDS, False)
    print('ID:', ID_COL, '| text:', TR_TEXT, '/', TE_TEXT, '| caption:', TR_CAP, '/', TE_CAP)
    
    missing = [l for l in LABELS if l not in train_df.columns]
    assert not missing, f'Label tidak ditemukan: {missing}'
    
    POS_TOKENS = {'1', '1.0', 'relevan', 'ya', 'yes', 'true', 'y'}
    
    
    def to_binary(s):
        if pd.api.types.is_numeric_dtype(s):
            return s.fillna(0).astype(float).round().clip(0, 1).astype(int)
        return s.astype(str).str.strip().str.lower().isin(POS_TOKENS).astype(int)
    
    
    for l in LABELS:
        train_df[l] = to_binary(train_df[l])
    Y = train_df[LABELS].values.astype(np.int64)
    N_TRAIN, N_TEST = len(train_df), len(test_df)
    
    print(f'Train rows: {N_TRAIN} (expected 23297) | Test rows: {N_TEST} (expected 34687)')
    if N_TRAIN != 23297:
        print('WARNING: jumlah baris train berbeda dari 23.297 - pastikan tidak ada baris yang dibuang.')
    print(pd.DataFrame({'prevalence_%': (Y.mean(0) * 100).round(2), 'positives': Y.sum(0)}, index=LABELS))
    print('Cardinality-0 rows:', int((Y.sum(1) == 0).sum()))
else:
    print("[lewati] RUN_V7 = False")


### B.7 Deobfuskasi (v7)

In [ ]:
if RUN_V7:
    ZW_RE = re.compile(r'[\u200b\u200c\u200d\u2060\ufeff\u00ad\u1160\u30b7]+')
    HOMOGLYPH_MAP = str.maketrans({
        'а': 'a', 'і': 'i', 'е': 'e', 'о': 'o', 'р': 'p', 'с': 'c', 'у': 'y', 'х': 'x',
        'к': 'k', 'т': 't', 'м': 'm', 'н': 'n', 'в': 'b', 'ѕ': 's', 'ј': 'j', 'ԁ': 'd',
        'А': 'a', 'І': 'i', 'Е': 'e', 'О': 'o', 'Р': 'p', 'С': 'c', 'У': 'y', 'Х': 'x',
        'К': 'k', 'Т': 't', 'М': 'm', 'Н': 'n', 'В': 'b', 'Ѕ': 's', 'Ј': 'j',
        'à': 'a', 'á': 'a', 'â': 'a', 'ã': 'a', 'ä': 'a', 'å': 'a', 'ᴀ': 'a',
        'è': 'e', 'é': 'e', 'ê': 'e', 'ë': 'e', 'ì': 'i', 'í': 'i', 'î': 'i', 'ï': 'i',
        'ò': 'o', 'ó': 'o', 'ô': 'o', 'õ': 'o', 'ö': 'o', 'ù': 'u', 'ú': 'u', 'û': 'u', 'ü': 'u',
    })
    URL_RE = re.compile(r'https?://\S+|www\.\S+')
    MENTION_RE = re.compile(r'@\w+')
    WS_RE = re.compile(r'\s+')
    TOKEN_RE = re.compile(r'[a-z0-9]+')
    DOT_SEP_RE = re.compile(r'\b(?:[a-z0-9]\.){2,}[a-z0-9]\b')
    SPACE_SEP_RE = re.compile(r'(?<![a-z0-9])(?:[a-z]\s+){2,}[a-z](?![a-z0-9])')
    
    
    def load_wordlist_csv(path, preferred=('lemma', 'lema', 'kata', 'word', 'entri', 'entry', 'slang', 'formal')):
        words = set()
        if path is None or not os.path.isfile(path):
            return words
        df = pd.read_csv(path, dtype=str, on_bad_lines='skip', encoding_errors='ignore')
        cols = [c for c in df.columns if str(c).strip().lower() in preferred]
        if not cols:
            best, best_score = None, -1.0
            for c in df.columns:
                v = df[c].dropna().astype(str).head(3000)
                if len(v) == 0:
                    continue
                score = v.str.fullmatch(r'[A-Za-z][A-Za-z -]{0,30}').mean()
                if score > best_score:
                    best, best_score = c, score
            cols = [best] if best is not None else []
        for c in cols:
            for val in df[c].dropna().astype(str):
                toks = re.findall(r'[a-z]+', val.lower())
                words.update(t for t in toks if len(t) >= 2)
                joined = ''.join(toks)
                if 2 <= len(joined) <= 30:
                    words.add(joined)
        print(f'  {os.path.basename(path)}: kolom {cols} -> {len(words):,} kata')
        return words
    
    
    SEED_LEXICON = {
        'ktp', 'kk', 'sim', 'stnk', 'bpkb', 'npwp', 'skck', 'pdam', 'pln', 'bpjs', 'kis', 'kip', 'pkh', 'bansos', 'blt',
        'pungli', 'ppdb', 'sekolah', 'smkn', 'sman', 'smpn', 'sdn', 'puskesmas', 'rsud', 'satpol', 'dishub', 'polres',
        'polsek', 'polisi', 'damkar', 'tps', 'tpa', 'dukcapil', 'kelurahan', 'kecamatan', 'pemda', 'pemkot', 'pemkab',
        'mbg', 'kopdes', 'kdmp', 'desil', 'pbi', 'dtks', 'pip', 'odol', 'jlnt', 'lrt', 'mrt', 'krl', 'transjakarta',
        'tije', 'jaki', 'jukir', 'qris', 'ormas', 'tawuran', 'begal', 'tilang', 'razia', 'jambret', 'trotoar', 'drainase',
        'plafon', 'honorer', 'pppk', 'cpns', 'stunting', 'rujukan', 'pasien', 'banjir', 'sampah', 'macet', 'kemacetan',
        'kemiskinan', 'pendidikan', 'mahasiswa', 'pelajar', 'kurikulum', 'korupsi', 'koruptor', 'preman', 'premanisme',
        'prabowo', 'jokowi', 'gibran', 'jakarta', 'bandung', 'surabaya', 'medan', 'bekasi', 'tangerang', 'depok', 'bogor',
    }
    PROTECTED_WORDS = {
        'type', 'next', 'guys', 'online', 'update', 'driver', 'truck', 'kitchen', 'slot', 'viral', 'hoax', 'event',
        'nice', 'boss', 'live', 'post', 'drop', 'link', 'info', 'open', 'free', 'full', 'auto', 'real', 'fake', 'good',
        'best', 'love', 'like', 'time', 'game', 'play', 'work', 'home', 'food', 'kids', 'news', 'team', 'user', 'httpurl',
        'aya', 'mah', 'teh', 'kitu', 'atuh', 'teu', 'lain', 'eta', 'keneh', 'kamana', 'sakola', 'karunya', 'pun', 'rek',
        'iki', 'ae', 'ta', 'ora', 'gak', 'nggak', 'udah', 'udh', 'sdh', 'blm', 'tdk', 'utk', 'dgn', 'krn', 'yg', 'klu',
        'klo', 'bgt', 'banget', 'gercep', 'mantap', 'keren', 'amin', 'allah',
    }
    DIRECT_SLANG_MAP = {
        'mb9': 'mbg', 'b4n505': 'bansos', '53k0l4h': 'sekolah', '54mp4h': 'sampah', 'p4513n': 'pasien',
        'j4l4n': 'jalan', 'd4l4n3': 'jalannya', 'p3nd1d1k4n': 'pendidikan', 'k3m15k1n4n': 'kemiskinan',
        'm15k1n': 'miskin', 't4wur4n': 'tawuran', 'b394l': 'begal', 'b364l': 'begal', 'j4mbr3t': 'jambret',
        'pr3m4n': 'preman', 'p4rk1r': 'parkir', 'l14r': 'liar', '53m094': 'semoga', '53k4l14n': 'sekalian',
        'm1cr0pl45t1c': 'mikroplastik', '3nt3k': 'habis', 'h1rup': 'hidup', 'put1h': 'putih', '45l1': 'asli',
        '53umur': 'seumur', 'b4ntu4n': 'bantuan', 'p3ny4lur4n': 'penyaluran', '54l4h': 'salah', '5454r4n': 'sasaran',
        'koprasi': 'koperasi',
    }
    
    print('Membangun leksikon v7 ...')
    LEXICON = set(SEED_LEXICON) | set(PROTECTED_WORDS)
    LEXICON |= load_wordlist_csv(KBBI_PATH)
    LEXICON |= load_wordlist_csv(COLLOQ_PATH)
    print(f'Total leksikon: {len(LEXICON):,} kata')
    
    
    def merge_spaced_letters(m):
        letters = m.group(0).split()
        joined = ''.join(letters)
        if joined in LEXICON:
            return ' ' + joined + ' '
        n = len(joined)
        for k in range(3, n - 2):
            w1, w2 = joined[:k], joined[k:]
            if w1 in LEXICON and w2 in LEXICON:
                return f' {w1} {w2} '
        return ' ' + joined + ' '
    
    
    def basic_clean(s):
        if not isinstance(s, str):
            return ''
        s = html.unescape(s)
        s = ZW_RE.sub('', s)
        s = s.translate(HOMOGLYPH_MAP)
        s = unicodedata.normalize('NFKC', s).lower()
        s = ZW_RE.sub('', s)
        s = s.translate(HOMOGLYPH_MAP)
        s = URL_RE.sub(' httpurl ', s)
        s = MENTION_RE.sub(' @user ', s)
        s = DOT_SEP_RE.sub(lambda m: m.group(0).replace('.', ''), s)
        s = SPACE_SEP_RE.sub(merge_spaced_letters, s)
        return WS_RE.sub(' ', s).strip()
    
    
    SUFFIXES = ('nya', 'lah', 'kah', 'pun', 'ku', 'mu', 'kan', 'an', 'i')
    PREFIXES = (('meng', ''), ('meng', 'k'), ('meny', 's'), ('mem', ''), ('mem', 'p'), ('men', ''), ('men', 't'),
                ('me', ''), ('peng', ''), ('peng', 'k'), ('peny', 's'), ('pem', ''), ('pem', 'p'), ('pen', ''),
                ('pen', 't'), ('per', ''), ('pe', ''), ('ber', ''), ('be', ''), ('ter', ''), ('di', ''), ('ke', ''),
                ('se', ''))
    LEET = str.maketrans({'0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's', '6': 'g', '7': 't', '8': 'b', '9': 'g'})
    LEET_ALT = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '6': 'g', '7': 't', '8': 'b', '9': 'g'})
    
    
    def morph_stems(w):
        stems = {w}
        for _ in range(2):
            for st in list(stems):
                for s in SUFFIXES:
                    if st.endswith(s) and len(st) - len(s) >= 3:
                        stems.add(st[:-len(s)])
        for st in list(stems):
            for p, r in PREFIXES:
                if st.startswith(p) and len(st) - len(p) >= 2:
                    cand = r + st[len(p):]
                    if len(cand) >= 3:
                        stems.add(cand)
        stems.discard(w)
        return stems
    
    
    class CaesarDecoder:
        def __init__(self, lexicon, corpus_counter, min_len=4, plain_min_count=30, frequent_min_count=25):
            self.lex = lexicon
            self.cnt = corpus_counter
            self.min_len = min_len
            self.plain_min_count = plain_min_count
            self.frequent = {w for w, c in corpus_counter.items() if c >= frequent_min_count and w.isalpha()}
            letters = 'abcdefghijklmnopqrstuvwxyz'
            self.tables = [(k, str.maketrans(letters, letters[k:] + letters[:k])) for k in range(1, 26)]
            self.cache = {}
            self.ls_cache = {}
            self.n_changed = 0
    
        def lexical_strength(self, w):
            if w in self.ls_cache:
                return self.ls_cache[w]
            s = 0
            if w in self.lex:
                s = 3
            elif len(w) >= 5 and any(st in self.lex for st in morph_stems(w)):
                s = 2
            self.ls_cache[w] = s
            return s
    
        def decode_word(self, w):
            if w in self.cache:
                return self.cache[w]
            if w in DIRECT_SLANG_MAP:
                self.cache[w] = DIRECT_SLANG_MAP[w]
                return self.cache[w]
            if w in PROTECTED_WORDS or len(w) < self.min_len or not w.isalpha() or self.lexical_strength(w) > 0:
                self.cache[w] = w
                return w
            cnt_w = self.cnt.get(w, 0)
            if len(w) == 4 and cnt_w >= 15:
                self.cache[w] = w
                return w
            best, best_score = None, 0.0
            for _, tab in self.tables:
                c = w.translate(tab)
                s = self.lexical_strength(c)
                if s == 0:
                    continue
                cnt_c = self.cnt.get(c, 0)
                if len(w) <= 4 and (s < 3 or cnt_c < max(20, 10 * max(cnt_w, 1))):
                    continue
                if s == 2 and cnt_c < 3:
                    continue
                score = s * 10.0 + math.log1p(cnt_c)
                if score > best_score:
                    best, best_score = c, score
            res = w
            if best is not None:
                if not (cnt_w >= self.plain_min_count and cnt_w >= self.cnt.get(best, 0)):
                    res = best
            self.cache[w] = res
            return res
    
        def decode_token(self, tok):
            if tok in DIRECT_SLANG_MAP:
                return DIRECT_SLANG_MAP[tok]
            if tok.isdigit():
                return tok
            if tok.isalpha():
                return self.decode_word(tok)
            if tok.endswith('2') and tok[:-1].isalpha() and len(tok[:-1]) >= 3:
                return self.decode_word(tok[:-1])
            for table in (LEET, LEET_ALT):
                t2 = tok.translate(table)
                if t2.isalpha() and len(t2) >= 3:
                    if t2 in self.lex:
                        return t2
                    d = self.decode_word(t2)
                    if self.lexical_strength(d) >= 2:
                        return d
            return tok
    
        def decode_text(self, s):
            def repl(m):
                tok = m.group(0)
                out = self.decode_token(tok)
                if out != tok:
                    self.n_changed += 1
                return out
            return TOKEN_RE.sub(repl, s)
    
    
    for df, tcol, ccol in [(train_df, TR_TEXT, TR_CAP), (test_df, TE_TEXT, TE_CAP)]:
        df['text_clean'] = df[tcol].map(basic_clean)
        df['cap_clean'] = df[ccol].map(basic_clean) if ccol is not None else ''
    
    corpus_counter = Counter()
    for s in pd.concat([train_df['text_clean'], test_df['text_clean']]):
        corpus_counter.update(TOKEN_RE.findall(s))
    for s in pd.concat([train_df['cap_clean'], test_df['cap_clean']]).drop_duplicates():
        corpus_counter.update(TOKEN_RE.findall(s))
    print(f'Vocab korpus bersih (pasca 5-layer clean): {len(corpus_counter):,} token unik')
    
    decoder = CaesarDecoder(LEXICON, corpus_counter)
else:
    print("[lewati] RUN_V7 = False")


### B.8 Jalankan dekode (v7)

In [ ]:
if RUN_V7:
    t0 = time.time()
    cap_cache = {}
    
    
    def decode_caption(c):
        if c not in cap_cache:
            cap_cache[c] = decoder.decode_text(c)
        return cap_cache[c]
    
    
    for df in (train_df, test_df):
        df['text_dec'] = df['text_clean'].map(decoder.decode_text)
        df['cap_dec'] = df['cap_clean'].map(decode_caption)
    print(f'Dekode selesai dalam {time.time() - t0:.1f}s | token dipulihkan: {decoder.n_changed:,}')
    
    CHECKS = {'wlshfhuhu': 'pelayanan', 'fvzaln': 'simnya', 'kvtpzpsp': 'domisili', 'alwlv': 'palak',
              'ufxtpfs': 'pasokan', 'xmujmzqbipciv': 'pemberitahuan', 'zrarzonx': 'menembak'}
    n_ok = 0
    for enc_word, expected in CHECKS.items():
        got = decoder.decode_word(enc_word)
        ok = got == expected
        n_ok += int(ok)
        status = 'OK' if ok else 'MISS'
        print(f'  {enc_word:>15} -> {got:<15} (expected {expected:<14}) {status}')
    print(f'Verifikasi kata kunci: {n_ok}/{len(CHECKS)} terpulihkan')
    
    changed = train_df.loc[train_df['text_clean'] != train_df['text_dec'], ['text_clean', 'text_dec']].head(8)
    print(changed.to_string())
else:
    print("[lewati] RUN_V7 = False")


### B.10 Fold & pos_weight (v7)

In [ ]:
if RUN_V7:
    group_ids = np.zeros(N_TRAIN, dtype=np.int64)
    cap_to_gid = {}
    next_gid = 0
    n_twitter = 0
    for i, c in enumerate(train_df['cap_dec'].values):
        if isinstance(c, str) and c.strip():
            if c not in cap_to_gid:
                cap_to_gid[c] = next_gid
                next_gid += 1
            group_ids[i] = cap_to_gid[c]
        else:
            group_ids[i] = -1
            n_twitter += 1
    for i in np.where(group_ids == -1)[0]:
        group_ids[i] = next_gid
        next_gid += 1
    train_df['post_id'] = group_ids
    print(f'Postingan unik ber-caption: {len(cap_to_gid)} | baris tanpa caption (grup unik): {n_twitter}')
    
    # Custom Balanced Multi-Label GroupKFold (Mencegah penumpukan 13.444 baris di Fold 4!)
    gid_to_rows = defaultdict(list)
    for i, g in enumerate(group_ids):
        gid_to_rows[int(g)].append(i)
    
    sorted_gids = sorted(gid_to_rows.keys(), key=lambda g: (-len(gid_to_rows[g]), g))
    target_rows = N_TRAIN / CFG.n_splits
    tot_pos = Y.sum(0).astype(np.float64)
    target_pos = tot_pos / CFG.n_splits
    
    fold_rows = np.zeros(CFG.n_splits, dtype=np.int64)
    fold_pos = np.zeros((CFG.n_splits, N_LABELS), dtype=np.float64)
    folds = np.full(N_TRAIN, -1, dtype=np.int64)
    
    for g in sorted_gids:
        idxs = gid_to_rows[g]
        g_len = len(idxs)
        g_pos = Y[idxs].sum(0)
        best_f, best_delta = 0, float('inf')
        for f in range(CFG.n_splits):
            old_r = ((fold_rows[f] - target_rows) / target_rows) ** 2
            new_r = ((fold_rows[f] + g_len - target_rows) / target_rows) ** 2
            old_p = np.sum(((fold_pos[f] - target_pos) / np.maximum(target_pos, 1.0)) ** 2)
            new_p = np.sum(((fold_pos[f] + g_pos - target_pos) / np.maximum(target_pos, 1.0)) ** 2)
            delta = float((new_r - old_r) * 4.0 + (new_p - old_p))
            if delta < best_delta:
                best_delta, best_f = delta, f
        fold_rows[best_f] += g_len
        fold_pos[best_f] += g_pos
        folds[idxs] = best_f
    
    train_df['fold'] = folds
    assert (folds >= 0).all()
    assert train_df.groupby('post_id')['fold'].nunique().max() == 1, 'Kebocoran grup antar fold!'
    print('Leak-free Balanced GroupKFold OK (setiap post_id hanya berada di 1 fold).')
    print('Rows per fold:', np.bincount(folds).tolist())
    print(pd.DataFrame({f'fold{f}': Y[folds == f].mean(0).round(4) for f in range(CFG.n_splits)}, index=LABELS))
    
    n_pos = Y.sum(0).astype(np.float64)
    POS_WEIGHT = np.clip(0.75 * np.sqrt((N_TRAIN - n_pos) / np.maximum(n_pos, 1.0)), CFG.pos_weight_min, CFG.pos_weight_max)
    POS_WEIGHT_T = torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)
    print(pd.Series(POS_WEIGHT.round(3), index=LABELS, name='pos_weight'))
else:
    print("[lewati] RUN_V7 = False")


### B.12 Tokenisasi, model, metrik (v7)

In [ ]:
if RUN_V7:
    def encode(tok, texts, caps, max_len, truncate=True):
        n = len(texts)
        res = [None] * n
        pair_idx = [i for i, c in enumerate(caps) if c]
        single_idx = [i for i, c in enumerate(caps) if not c]
        kw = dict(truncation='longest_first', max_length=max_len) if truncate else dict(truncation=False)
        sep_id = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
        if pair_idx:
            p_texts = [texts[i] for i in pair_idx]
            p_caps = [caps[i] for i in pair_idx]
            e = tok(p_texts, p_caps, **kw)
            keys = list(e.keys())
            for k, i in enumerate(pair_idx):
                d = {key: e[key][k] for key in keys}
                ids = d['input_ids']
                c_len = (ids.index(sep_id) + 1) if (sep_id is not None and sep_id in ids) else len(ids)
                d['comment_mask'] = [1 if pos < c_len else 0 for pos in range(len(ids))]
                res[i] = d
        if single_idx:
            s_texts = [texts[i] for i in single_idx]
            e = tok(s_texts, **kw)
            keys = list(e.keys())
            for k, i in enumerate(single_idx):
                d = {key: e[key][k] for key in keys}
                d['comment_mask'] = [1] * len(d['input_ids'])
                res[i] = d
        keys = list(res[0].keys())
        return {key: [r[key] for r in res] for key in keys}
    
    
    class PairDataset(Dataset):
        def __init__(self, enc, indices, labels=None):
            self.enc, self.indices, self.labels = enc, np.asarray(indices), labels
    
        def __len__(self):
            return len(self.indices)
    
        def __getitem__(self, i):
            idx = self.indices[i]
            item = {k: v[idx] for k, v in self.enc.items()}
            if self.labels is not None:
                item['labels'] = self.labels[idx]
            return item
    
    
    def make_collate(tok):
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
    
        def collate(batch):
            bs = len(batch)
            max_l = max(len(b['input_ids']) for b in batch)
            max_l = int(math.ceil(max_l / 8.0) * 8)
    
            input_ids = torch.full((bs, max_l), pad_id, dtype=torch.long)
            attention_mask = torch.zeros((bs, max_l), dtype=torch.long)
            comment_mask = torch.zeros((bs, max_l), dtype=torch.float32)
            has_tt = 'token_type_ids' in batch[0]
            token_type_ids = torch.zeros((bs, max_l), dtype=torch.long) if has_tt else None
    
            for i, b in enumerate(batch):
                ids = b['input_ids']
                L = len(ids)
                input_ids[i, :L] = torch.tensor(ids, dtype=torch.long)
                attention_mask[i, :L] = torch.tensor(b['attention_mask'], dtype=torch.long)
                cm = b.get('comment_mask')
                if cm is not None:
                    comment_mask[i, :len(cm)] = torch.tensor(cm, dtype=torch.float32)
                else:
                    comment_mask[i, :L] = 1.0
                if has_tt:
                    token_type_ids[i, :L] = torch.tensor(b['token_type_ids'], dtype=torch.long)
    
            out = {'input_ids': input_ids, 'attention_mask': attention_mask, 'comment_mask': comment_mask}
            if has_tt:
                out['token_type_ids'] = token_type_ids
            if 'labels' in batch[0]:
                out['labels'] = torch.tensor(np.stack([b['labels'] for b in batch]), dtype=torch.float32)
            return out
    
        return collate
    
    
    def make_loader(enc, indices, tok, labels=None, shuffle=False, sort_by_len=False, batch_size=32):
        indices = np.asarray(indices)
        if sort_by_len:
            lens = np.array([len(enc['input_ids'][i]) for i in indices])
            indices = indices[np.argsort(lens, kind='stable')]
        ds = PairDataset(enc, indices, labels)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=make_collate(tok),
                            num_workers=0, pin_memory=USE_AMP, drop_last=False)
        return loader, indices
    
    
    class MultiLabelModel(nn.Module):
        def __init__(self, name, n_labels, dropout_ps, prior=None):
            super().__init__()
            config = AutoConfig.from_pretrained(name)
            self.backbone = AutoModel.from_pretrained(name, config=config)
            hidden = config.hidden_size
            # Gated Dual-Segment Pooling: [CLS, Mean(Comment), Max(Comment), Gate * Mean(Caption)] -> 4 * hidden
            self.feat_dim = 4 * hidden
            self.cap_gate = nn.Sequential(
                nn.Linear(2 * hidden, hidden),
                nn.Tanh(),
                nn.Linear(hidden, hidden),
                nn.Sigmoid(),
            )
            self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropout_ps])
            self.head = nn.Linear(self.feat_dim, n_labels)
            nn.init.normal_(self.head.weight, std=0.02)
            if prior is not None:
                p = np.clip(np.asarray(prior, dtype=np.float64), 1e-4, 1 - 1e-4)
                self.head.bias.data = torch.tensor(np.log(p / (1 - p)), dtype=torch.float32)
            else:
                nn.init.zeros_(self.head.bias)
    
        def forward(self, input_ids, attention_mask, token_type_ids=None, comment_mask=None):
            kw = dict(input_ids=input_ids, attention_mask=attention_mask)
            if token_type_ids is not None:
                kw['token_type_ids'] = token_type_ids
            hs = self.backbone(**kw).last_hidden_state.float()
            cls = hs[:, 0]
            full_m = attention_mask.unsqueeze(-1).to(hs.dtype)
            if comment_mask is not None:
                comm_m = (comment_mask * attention_mask.float()).unsqueeze(-1).to(hs.dtype)
                cap_m = ((1.0 - comment_mask) * attention_mask.float()).unsqueeze(-1).to(hs.dtype)
            else:
                comm_m = full_m
                cap_m = full_m
            mean_comm = (hs * comm_m).sum(1) / comm_m.sum(1).clamp(min=1e-6)
            hs_masked = hs.masked_fill(comm_m == 0, -1e4)
            max_comm = hs_masked.max(1).values
            mean_cap = (hs * cap_m).sum(1) / cap_m.sum(1).clamp(min=1e-6)
            gate = self.cap_gate(torch.cat([cls, mean_comm], dim=-1))
            feat = torch.cat([cls, mean_comm, max_comm, gate * mean_cap], dim=-1)
            logits = torch.stack([self.head(d(feat)) for d in self.dropouts], dim=0).mean(0)
            return logits
    
    
    NO_DECAY = ('bias', 'LayerNorm.weight', 'LayerNorm.bias', 'layer_norm', 'layernorm')
    
    
    def get_param_groups(model):
        groups = []
    
        def add(named, lr):
            dec = [p for n, p in named if p.requires_grad and not any(nd in n for nd in NO_DECAY)]
            nod = [p for n, p in named if p.requires_grad and any(nd in n for nd in NO_DECAY)]
            if dec:
                groups.append({'params': dec, 'lr': lr, 'weight_decay': CFG.weight_decay})
            if nod:
                groups.append({'params': nod, 'lr': lr, 'weight_decay': 0.0})
    
        bb = model.backbone
        enc = getattr(bb, 'encoder', None)
        layers = list(enc.layer) if enc is not None and hasattr(enc, 'layer') else []
        assigned = set()
        for depth, layer in enumerate(reversed(layers)):
            named = list(layer.named_parameters())
            add(named, CFG.lr * (CFG.llrd ** depth))
            assigned |= {id(p) for _, p in named}
        rest = [(n, p) for n, p in bb.named_parameters() if id(p) not in assigned]
        add(rest, CFG.lr * (CFG.llrd ** len(layers)))
        add(list(model.cap_gate.named_parameters()) + list(model.head.named_parameters()), CFG.head_lr)
        return groups
    
    
    @torch.no_grad()
    def predict(model, loader):
        model.eval()
        preds = []
        for b in loader:
            b.pop('labels', None)
            b = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(**b)
            preds.append(torch.sigmoid(logits.float()).cpu().numpy())
        return np.concatenate(preds, 0)
    
    
    THR_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
    
    
    def macro_f1_grid(y, p, thr=THR_GRID):
        y = np.asarray(y).astype(np.int64)
        p = np.asarray(p, dtype=np.float64)
        thr = np.asarray(thr, dtype=np.float64)
        order = np.argsort(-p, kind='stable')
        ps, ys = p[order], y[order]
        n_pred = np.searchsorted(-ps, -thr, side='right')
        ctp = np.concatenate([[0], np.cumsum(ys)])
        tp = ctp[n_pred]
        fp = n_pred - tp
        n_pos_true = ys.sum()
        fn = n_pos_true - tp
        tn = len(ys) - n_pos_true - fp
        f1_pos = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        f1_neg = 2 * tn / np.maximum(2 * tn + fp + fn, 1)
        return (f1_pos + f1_neg) / 2.0
    
    
    def macro_f1_at(y, p, t):
        return float(macro_f1_grid(y, p, np.array([t]))[0])
    
    
    def per_pillar_best(y, P):
        rows = []
        for j in range(P.shape[1]):
            s = macro_f1_grid(y[:, j], P[:, j])
            k = int(np.argmax(s))
            rows.append((float(THR_GRID[k]), float(s[k])))
        return rows
    
    
    def maybe_save_global_best(state, backbone, fold, score, max_len):
        meta = {'score': -1.0}
        if os.path.isfile(BEST_META_PATH):
            with open(BEST_META_PATH) as fh:
                meta = json.load(fh)
        if score > meta.get('score', -1.0):
            torch.save(state, BEST_CKPT_PATH)
            meta = {'score': float(score), 'backbone': backbone, 'fold': int(fold), 'max_len': int(max_len),
                    'pooling': 'gated_dual_segment', 'labels': LABELS}
            with open(BEST_META_PATH, 'w') as fh:
                json.dump(meta, fh, indent=2)
            print(f'  >> best_model.pth diperbarui ({backbone}, fold {fold}, score {score:.4f})')
        for p in glob.glob(os.path.join(CFG.work_dir, '**', '*.pth'), recursive=True):
            if os.path.abspath(p) != os.path.abspath(BEST_CKPT_PATH):
                os.remove(p)
else:
    print("[lewati] RUN_V7 = False")


### B.14 Training per backbone (v7)

In [ ]:
if RUN_V7:
    BACKBONE_HOURS = {}
    
    
    def model_tag(name, seed_offset=0):
        tag = name.split('/')[-1]
        return tag if seed_offset == 0 else f'{tag}_seed{CFG.seed + 100 * seed_offset}'
    
    
    def choose_max_len(name, tok, tr_texts, tr_caps, te_texts, te_caps):
        config = AutoConfig.from_pretrained(name)
        limit = int(getattr(config, 'max_position_embeddings', 512) or 512)
        if getattr(config, 'model_type', '') in ('roberta', 'xlm-roberta', 'camembert'):
            limit -= 2
        limit = min(limit, 512)
        sample_tr = list(range(0, len(tr_texts), max(1, len(tr_texts) // 2000)))
        sample_te = list(range(0, len(te_texts), max(1, len(te_texts) // 2000)))
        s_texts = [tr_texts[i] for i in sample_tr] + [te_texts[i] for i in sample_te]
        s_caps = [tr_caps[i] for i in sample_tr] + [te_caps[i] for i in sample_te]
        full = tok(s_texts, s_caps, truncation=False)
        lens = np.array([len(x) for x in full['input_ids']])
        del full
        ml = CFG.max_len
        if (lens > ml).any():
            need = int(math.ceil(lens.max() / 8.0) * 8)
            ml = max(CFG.max_len, min(need, CFG.max_len_cap))
        ml = min(ml, limit)
        print(f'  token median={np.median(lens):.0f} | p90={np.percentile(lens, 90):.0f} | '
              f'p99={np.percentile(lens, 99):.0f} | max_len dipakai = {ml}', flush=True)
        return ml
    
    
    def initial_accum(name, ml):
        big = any(k in name.lower() for k in ('large', '522m'))
        if not big or GPU_MEM_GB >= 70:
            return 1
        return 1 if ml <= 256 else 2
    
    
    def is_oom(e):
        return 'out of memory' in str(e).lower()
    
    
    def free_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    
    def train_one_fold(name, fold, seed, enc_tr, tok, ml, accum, te_loader, te_order):
        f_start = time.time()
        seed_everything(seed + fold)
        Yf = Y.astype(np.float32)
        tr_idx = np.where(folds != fold)[0]
        va_idx = np.where(folds == fold)[0]
        micro_bs = max(1, CFG.batch_size // accum)
        tr_loader, _ = make_loader(enc_tr, tr_idx, tok, labels=Yf, shuffle=True, batch_size=micro_bs)
        va_loader, va_order = make_loader(enc_tr, va_idx, tok, sort_by_len=True, batch_size=CFG.eval_batch_size)
    
        model = MultiLabelModel(name, N_LABELS, CFG.dropout_ps, prior=Y[tr_idx].mean(0)).to(DEVICE)
        if fold == 0:
            n_params = sum(p.numel() for p in model.backbone.parameters()) / 1e6
            print(f'  params backbone={n_params:.0f}M | fitur head={model.feat_dim} dim | '
                  f'micro-batch={micro_bs} x accum={accum} = batch efektif {micro_bs * accum} | max_len={ml}', flush=True)
        optimizer = torch.optim.AdamW(get_param_groups(model), lr=CFG.lr, eps=1e-6)
        n_batches = len(tr_loader)
        steps_per_epoch = math.ceil(n_batches / accum)
        total_steps = steps_per_epoch * CFG.epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT_T)
    
        best_score, best_state, best_val = -1.0, None, None
        for epoch in range(CFG.epochs):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            running, n_seen = 0.0, 0
            for step, batch in enumerate(tr_loader):
                labels = batch.pop('labels').to(DEVICE, non_blocking=True)
                batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    logits = model(**batch)
                loss = loss_fn(logits.float(), labels)
                scaler.scale(loss / accum).backward()
                if (step + 1) % accum == 0 or (step + 1) == n_batches:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()
                running += loss.item() * labels.size(0)
                n_seen += labels.size(0)
                if (step + 1) in (1, 100, 200, n_batches):
                    print(f'    [fold {fold} ep {epoch+1}] step {step+1}/{n_batches} | loss={running/max(n_seen,1):.4f}', flush=True)
    
            val_sorted = predict(model, va_loader)
            val = np.empty((len(va_idx), N_LABELS), dtype=np.float32)
            val[np.searchsorted(va_idx, va_order)] = val_sorted
            scores = per_pillar_best(Y[va_idx], val)
            score = float(np.mean([s for _, s in scores]))
            print(f'  fold {fold} ep {epoch + 1}/{CFG.epochs} | loss {running / max(n_seen, 1):.4f} | '
                  f'val mean macroF1 {score:.4f} | min pilar {min(s for _, s in scores):.4f} | '
                  f'{(time.time() - f_start) / 60:.1f} min', flush=True)
            if score > best_score:
                best_score, best_val = score, val
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    
        model.load_state_dict(best_state)
        te_sorted = predict(model, te_loader)
        te = np.empty_like(te_sorted)
        te[te_order] = te_sorted
        del model, optimizer, scheduler, scaler, tr_loader, va_loader
        free_cuda()
        return best_score, best_val, best_state, te
    
    
    def train_backbone(name, seed_offset=0):
        tag = model_tag(name, seed_offset)
        seed = CFG.seed + 100 * seed_offset
        oof_path = os.path.join(PRED_DIR, f'oof_{tag}.npy')
        test_path = os.path.join(PRED_DIR, f'test_{tag}.npy')
        partial_path = os.path.join(PRED_DIR, f'partial_{tag}.npz')
        if os.path.isfile(oof_path) and os.path.isfile(test_path):
            print(f'[{tag}] prediksi sudah ada di {PRED_DIR} - dimuat ulang.', flush=True)
            return np.load(oof_path), np.load(test_path)
    
        t_start = time.time()
        print('=' * 90)
        print(f'Backbone: {name} (seed {seed}) -> tag {tag}', flush=True)
        tok = AutoTokenizer.from_pretrained(name)
        tr_texts, tr_caps = train_df['text_dec'].tolist(), train_df['cap_dec'].tolist()
        te_texts, te_caps = test_df['text_dec'].tolist(), test_df['cap_dec'].tolist()
    
        ml = choose_max_len(name, tok, tr_texts, tr_caps, te_texts, te_caps)
        print(f'  Tokenizing train ({N_TRAIN}) & test ({N_TEST}) ...', flush=True)
        enc_tr = encode(tok, tr_texts, tr_caps, ml)
        enc_te = encode(tok, te_texts, te_caps, ml)
        te_loader, te_order = make_loader(enc_te, np.arange(N_TEST), tok, sort_by_len=True, batch_size=CFG.eval_batch_size)
    
        oof = np.zeros((N_TRAIN, N_LABELS), dtype=np.float32)
        test_pred = np.zeros((N_TEST, N_LABELS), dtype=np.float32)
        done = []
        if os.path.isfile(partial_path):
            z = np.load(partial_path)
            oof, test_pred, done = z['oof'], z['test'], z['done'].tolist()
            print(f'  Resume dari fold yang sudah selesai: {done}', flush=True)
    
        for fold in range(CFG.n_splits):
            if fold in done:
                continue
            accum = initial_accum(name, ml)
            while True:
                oom = False
                try:
                    score, val, state, te = train_one_fold(name, fold, seed, enc_tr, tok, ml, accum, te_loader, te_order)
                except RuntimeError as e:
                    if not is_oom(e):
                        raise
                    oom = True
                if not oom:
                    break
                free_cuda()
                accum *= 2
                if accum > 8:
                    raise RuntimeError(f'OOM persisten pada {name} bahkan dengan grad accumulation 8')
                print(f'  OOM -> grad accumulation dinaikkan ke {accum}', flush=True)
    
            va_idx = np.where(folds == fold)[0]
            oof[va_idx] = val
            test_pred += te / CFG.n_splits
            maybe_save_global_best(state, tag, fold, score, ml)
            del state
            free_cuda()
            done.append(fold)
            np.savez(partial_path, oof=oof, test=test_pred, done=np.array(done, dtype=np.int64))
    
        np.save(oof_path, oof)
        np.save(test_path, test_pred)
        if os.path.isfile(partial_path):
            os.remove(partial_path)
        hours = (time.time() - t_start) / 3600.0
        BACKBONE_HOURS[tag] = hours
        scores = per_pillar_best(Y, oof)
        mean_f1 = float(np.mean([s for _, s in scores]))
        print(f'[{tag}] OOF mean macroF1 (thr per pilar) = {mean_f1:.4f} | {hours:.2f} jam', flush=True)
        for l, (t, s) in zip(LABELS, scores):
            flag = 'PASS' if s >= CFG.target_f1 else 'FAIL'
            print(f'    {l:<26} thr={t:.2f}  macroF1={s:.4f}  {flag}')
        return oof, test_pred
else:
    print("[lewati] RUN_V7 = False")


### B.16 Stacking & optimasi ensemble (v7)

In [ ]:
if RUN_V7:
    KW_SETS = {
        0: {'jalan', 'jembatan', 'rusak', 'berlubang', 'aspal', 'trotoar', 'drainase', 'tanggul', 'roboh', 'rubuh', 'plafon', 'atap', 'halte', 'tol', 'infrastruktur', 'fasilitas', 'bangunan', 'gedung', 'proyek'},
        1: {'bpjs', 'pbi', 'puskesmas', 'rsud', 'pasien', 'berobat', 'tandu', 'nandu', 'dokter', 'medis', 'kesehatan', 'obat', 'sakit', 'virus', 'covid', 'stunting', 'gizi', 'imunisasi', 'vaksin'},
        2: {'rujukan', 'ktp', 'kk', 'sim', 'stnk', 'bpkb', 'dukcapil', 'izin', 'perizinan', 'pajak', 'pungli', 'pengawasan', 'mpp', 'administrasi', 'birokrasi', 'kecamatan', 'kelurahan', 'desa', 'pelayanan', 'daftar', 'antre', 'kuota', 'cetak'},
        3: {'sampah', 'tps', 'tpa', 'banjir', 'limbah', 'polusi', 'kotor', 'bau', 'selokan', 'got', 'kali', 'sungai', 'plastik', 'cemar', 'kebersihan', 'lingkungan', 'kumuh', 'hijau', 'pohon'},
        4: {'parkir', 'jukir', 'liar', 'tawuran', 'razia', 'satpol', 'tertib', 'ketertiban', 'gusur', 'miras', 'petasan', 'knalpot', 'demo', 'kericuhan', 'ricuh', 'ormas', 'trotoar', 'pedagang', 'pkl'},
        5: {'macet', 'kemacetan', 'lalin', 'angkot', 'transjakarta', 'tije', 'lrt', 'mrt', 'krl', 'stasiun', 'terminal', 'jalur', 'odol', 'truk', 'klakson', 'ugal', 'kendaraan', 'motor', 'mobil', 'transportasi', 'sopir', 'supir', 'penumpang', 'tilang'},
        6: {'bansos', 'bantuan', 'mbg', 'kopdes', 'kdmp', 'koperasi', 'desil', 'pkh', 'blt', 'dtks', 'kpm', 'kemiskinan', 'miskin', 'sejahtera', 'kemensos', 'sosial', 'sembako', 'dana', 'tunai'},
        7: {'begal', 'jambret', 'maling', 'curanmor', 'pencuri', 'perampok', 'kriminal', 'kejahatan', 'polisi', 'polsek', 'polres', 'tangkap', 'senjata', 'sajam', 'pembunuhan', 'pemerasan', 'judi', 'slot', 'korupsi', 'koruptor', 'aman', 'keamanan', 'patroli', 'teror'},
        8: {'sekolah', 'pendidikan', 'sman', 'smkn', 'sdn', 'smpn', 'guru', 'mahasiswa', 'pelajar', 'siswa', 'kurikulum', 'ujian', 'belajar', 'kampus', 'universitas', 'pip', 'kip', 'beasiswa', 'ppdb', 'zonasi', 'kelas'},
    }
    ALL_DOMAIN_KW = set().union(*KW_SETS.values())
    FP_TRAP_FOOD = {'pentol', 'bakso', 'cilok', 'kecap'}
    FP_TRAP_TOTO = {'toto', 'kitchen'}
    PRAISE_NOISE = {'amin', 'semoga', 'gercep', 'mantap', 'keren', 'hayooooo', 'gemoy', 'wkwk', 'wkwkwk'}
    
    _TFIDF_CACHE = None
    
    
    def get_tfidf_and_domain_features():
        global _TFIDF_CACHE
        if _TFIDF_CACHE is not None:
            return _TFIDF_CACHE
        print('  Menghitung fitur TF-IDF bersih & fitur disambiguasi domain ...', flush=True)
        vec = FeatureUnion([
            ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=35000, sublinear_tf=True, min_df=3)),
            ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=35000, sublinear_tf=True, min_df=5)),
        ])
        tr_full = (train_df['text_dec'] + ' ' + train_df['cap_dec']).tolist()
        te_full = (test_df['text_dec'] + ' ' + test_df['cap_dec']).tolist()
        X_tr_tf = vec.fit_transform(tr_full)
        X_te_tf = vec.transform(te_full)
    
        tfidf_oof = np.zeros((N_TRAIN, N_LABELS), dtype=np.float64)
        tfidf_te = np.zeros((N_TEST, N_LABELS), dtype=np.float64)
        for j in range(N_LABELS):
            for f in range(CFG.n_splits):
                tr, va = folds != f, folds == f
                clf = LogisticRegression(C=1.5, max_iter=1000)
                clf.fit(X_tr_tf[tr], Y[tr, j])
                tfidf_oof[va, j] = clf.predict_proba(X_tr_tf[va])[:, 1]
                tfidf_te[:, j] += clf.predict_proba(X_te_tf)[:, 1] / CFG.n_splits
    
        def build_dom(df):
            rows = []
            for t_s, c_s in zip(df['text_dec'], df['cap_dec']):
                toks = set(TOKEN_RE.findall(t_s))
                cap_toks = set(TOKEN_RE.findall(c_s))
                n_words = len([w for w in toks if len(w) >= 2 and w not in {'user', 'httpurl'}])
                has_dom = bool(toks & ALL_DOMAIN_KW)
                is_short_noise = 1.0 if (n_words <= 3 and not has_dom) else 0.0
                is_food_joke = 1.0 if ((toks & FP_TRAP_FOOD) and not (toks & KW_SETS[7])) else 0.0
                is_toto_ad = 1.0 if ((toks & FP_TRAP_TOTO) and not (toks & {'sampah', 'banjir', 'bau', 'kotor'})) else 0.0
                is_praise = 1.0 if (n_words <= 4 and (toks & PRAISE_NOISE) and not has_dom) else 0.0
                has_mbg = 1.0 if (toks & {'mbg', 'kopdes', 'kdmp', 'bansos', 'desil', 'miskin', 'kemiskinan'}) else 0.0
                has_rujukan = 1.0 if (toks & {'rujukan', 'pengawasan', 'pungli', 'izin', 'perizinan', 'pajak', 'ktp', 'sim', 'stnk', 'mpp', 'dukcapil'}) else 0.0
                has_desil_pip = 1.0 if (toks & {'desil', 'pip', 'kip', 'beasiswa'}) else 0.0
                has_pbi_tandu = 1.0 if (toks & {'pbi', 'tandu', 'nandu', 'bpjs', 'puskesmas', 'pasien', 'berobat'}) else 0.0
                has_sch_col = 1.0 if ((cap_toks & {'sekolah', 'sman', 'smkn', 'sdn'}) and (cap_toks & {'roboh', 'rubuh', 'rusak', 'plafon', 'atap'}) and not is_short_noise) else 0.0
                kw_hits = [1.0 if (toks & KW_SETS[k]) else 0.0 for k in range(N_LABELS)]
                rows.append([is_short_noise, is_food_joke, is_toto_ad, is_praise,
                             has_mbg, has_rujukan, has_desil_pip, has_pbi_tandu, has_sch_col] + kw_hits)
            return np.array(rows, dtype=np.float64)
    
        _TFIDF_CACHE = (tfidf_oof, tfidf_te, build_dom(train_df), build_dom(test_df))
        return _TFIDF_CACHE
    
    
    def to_logit(p, eps=1e-5):
        p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
        return np.log(p / (1 - p))
    
    
    def stack_cross_pillar(oof_dict, test_dict, C=0.25):
        names = [n for n in oof_dict.keys() if np.mean([ s for _, s in per_pillar_best(Y, oof_dict[n]) ]) >= 0.85]
        if not names:
            names = list(oof_dict.keys())
        tfidf_oof, tfidf_te, dom_tr, dom_te = get_tfidf_and_domain_features()
        Xtr = np.concatenate([to_logit(oof_dict[n]) for n in names] + [to_logit(tfidf_oof), dom_tr], 1)
        Xte = np.concatenate([to_logit(test_dict[n]) for n in names] + [to_logit(tfidf_te), dom_te], 1)
        oof_meta = np.zeros((N_TRAIN, N_LABELS), dtype=np.float64)
        test_meta = np.zeros((N_TEST, N_LABELS), dtype=np.float64)
        for j in range(N_LABELS):
            for f in range(CFG.n_splits):
                tr, va = folds != f, folds == f
                clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
                clf.fit(Xtr[tr], Y[tr, j])
                oof_meta[va, j] = clf.predict_proba(Xtr[va])[:, 1]
                test_meta[:, j] += clf.predict_proba(Xte)[:, 1] / CFG.n_splits
            # Apply High-FP Trap Dampening on Food Jokes (Class 7) & Toto Kitchen (Class 3)
            if j == 7:
                oof_meta[:, j] *= np.where(dom_tr[:, 1] > 0, 0.25, 1.0)
                test_meta[:, j] *= np.where(dom_te[:, 1] > 0, 0.25, 1.0)
            elif j == 3:
                oof_meta[:, j] *= np.where(dom_tr[:, 2] > 0, 0.25, 1.0)
                test_meta[:, j] *= np.where(dom_te[:, 2] > 0, 0.25, 1.0)
        return oof_meta, test_meta
    
    
    def simplex_grid(m, step):
        n = int(round(1.0 / step))
        out = []
    
        def rec(prefix, remaining, k):
            if k == 1:
                out.append(prefix + [remaining])
                return
            for i in range(remaining + 1):
                rec(prefix + [i], remaining - i, k - 1)
    
        rec([], n, m)
        return np.array(out, dtype=np.float64) / n
    
    
    def default_weight_step(m):
        if m <= 3:
            return 0.05
        if m == 4:
            return 0.1
        if m <= 7:
            return 0.2
        return 0.25
    
    
    def optimize_pillar_ensemble(oof_dict, test_dict, y, weight_step=None, thr_grid=THR_GRID, verbose=True):
        # Filter out any unconverged backbone (< 0.85 Macro-F1)
        names = [n for n in oof_dict.keys() if np.mean([s for _, s in per_pillar_best(y, oof_dict[n])]) >= 0.85]
        if not names:
            names = list(oof_dict.keys())
        m = len(names)
        if weight_step is None:
            weight_step = default_weight_step(m)
        W = simplex_grid(m, weight_step)
        n_tr = len(y)
        n_te = len(next(iter(test_dict.values())))
        out = {'names': names, 'weights': np.zeros((N_LABELS, m)), 'thresholds': np.zeros(N_LABELS),
               'scores': np.zeros(N_LABELS), 'oof_prob': np.zeros((n_tr, N_LABELS)),
               'test_prob': np.zeros((n_te, N_LABELS))}
        for j in range(N_LABELS):
            O = np.stack([np.asarray(oof_dict[n][:, j], dtype=np.float64) for n in names], 0)
            T = np.stack([np.asarray(test_dict[n][:, j], dtype=np.float64) for n in names], 0)
            prior_j = float(y[:, j].mean())
            best_obj, best_s, best_w, best_t = -1.0, -1.0, W[0], 0.5
            for w in W:
                p_blend = w @ O
                s = macro_f1_grid(y[:, j], p_blend, thr_grid)
                # Prior-regularized threshold selection
                pred_prevs = np.array([(p_blend >= t).mean() for t in thr_grid])
                objs = s - 0.04 * np.abs(pred_prevs - prior_j)
                k = int(np.argmax(objs))
                if objs[k] > best_obj + 1e-9:
                    best_obj, best_s, best_w, best_t = float(objs[k]), float(s[k]), w, float(thr_grid[k])
            out['weights'][j] = best_w
            out['thresholds'][j] = best_t
            out['scores'][j] = best_s
            out['oof_prob'][:, j] = best_w @ O
            out['test_prob'][:, j] = best_w @ T
        out['oof_bin'] = (out['oof_prob'] >= out['thresholds'][None, :]).astype(int)
        out['test_bin'] = (out['test_prob'] >= out['thresholds'][None, :]).astype(int)
        if verbose:
            df = pd.DataFrame(out['weights'].round(2), index=LABELS, columns=names)
            df['thr'] = out['thresholds']
            df['macroF1'] = out['scores'].round(5)
            print(df.to_string())
        return out
    
    
    def run_ensemble(oof_dict, test_dict):
        # Also load existing v5/v6 neural models if available for Dual-View Ensembling!
        oof_all, test_all = {}, {}
        for k, v in oof_dict.items():
            sc = float(np.mean([s for _, s in per_pillar_best(Y, v)]))
            if sc >= 0.85:
                oof_all[k] = v
                test_all[k] = test_dict[k]
            else:
                print(f'  [SKIP] Model {k} diabaikan karena OOF Macro-F1 rendah ({sc:.4f} < 0.85)')
        sm_oof, sm_test = stack_cross_pillar(oof_all, test_all)
        oof_all['stack_v7'], test_all['stack_v7'] = sm_oof, sm_test
        st = per_pillar_best(Y, sm_oof)
        print(f'Stage-2 Meta-Stacker v7 OOF mean macroF1 = {np.mean([s for _, s in st]):.5f}')
        res = optimize_pillar_ensemble(oof_all, test_all, Y)
        return res, oof_all, test_all
    
    
    def honest_cv_scores(oof_dict, y):
        names = [n for n in oof_dict.keys() if np.mean([s for _, s in per_pillar_best(y, oof_dict[n])]) >= 0.85]
        pred = np.zeros_like(y)
        for f in range(CFG.n_splits):
            tr, va = folds != f, folds == f
            r = optimize_pillar_ensemble({n: oof_dict[n][tr] for n in names},
                                         {n: oof_dict[n][va] for n in names}, y[tr], verbose=False)
            pred[va] = r['test_bin']
        return np.array([macro_f1_at(y[:, j], pred[:, j].astype(float), 0.5) for j in range(N_LABELS)])
    
    
    def audit(res, honest=None):
        rows = []
        for j, l in enumerate(LABELS):
            r = {'pillar': l, 'oof_macroF1': round(float(res['scores'][j]), 5), 'thr': float(res['thresholds'][j])}
            if honest is not None:
                r['nested_cv_macroF1'] = round(float(honest[j]), 5)
            r['status'] = 'PASS' if res['scores'][j] >= CFG.target_f1 else 'FAIL'
            rows.append(r)
        print(pd.DataFrame(rows).to_string(index=False))
        mean_f1 = float(np.mean(res['scores']))
        n_pass = int((res['scores'] >= CFG.target_f1).sum())
        print(f'Mean macro F1 (OOF) = {mean_f1:.5f} | pilar >= {CFG.target_f1}: {n_pass}/{N_LABELS}')
        if honest is not None:
            print(f'Mean macro F1 (nested CV, estimasi jujur) = {float(np.mean(honest)):.5f}')
        return n_pass == N_LABELS
else:
    print("[lewati] RUN_V7 = False")


### B.18 Eksekusi training + iterasi otomatis (v7)

In [ ]:
if RUN_V7:
    OOF, TEST = {}, {}
    for name in CFG.base_backbones:
        tag = model_tag(name)
        OOF[tag], TEST[tag] = train_backbone(name)
    
    RESULT, OOF_ALL, TEST_ALL = run_ensemble(OOF, TEST)
    ALL_PASS = audit(RESULT)
    
    queue = [(b, 0) for b in CFG.extra_backbones]
    queue += [(b, k) for k in range(1, CFG.n_seed_variants + 1) for b in CFG.seed_variants]
    queue = [(b, s) for b, s in queue if model_tag(b, s) not in OOF]
    
    while not ALL_PASS and queue:
        est = max(BACKBONE_HOURS.values()) * 1.15 if BACKBONE_HOURS else 1.5
        if elapsed_hours() + est > CFG.time_budget_hours:
            print(f'Sisa waktu tidak cukup (terpakai {elapsed_hours():.2f} jam, estimasi {est:.2f} jam/backbone).')
            break
        name, s_off = queue.pop(0)
        tag = model_tag(name, s_off)
        failing = [l for j, l in enumerate(LABELS) if RESULT['scores'][j] < CFG.target_f1]
        print(f'Pilar < {CFG.target_f1}: {failing} -> menambah backbone: {tag}')
        try:
            OOF[tag], TEST[tag] = train_backbone(name, s_off)
        except Exception as e:
            print(f'  Gagal melatih {tag}: {e}')
            free_cuda()
            continue
        RESULT, OOF_ALL, TEST_ALL = run_ensemble(OOF, TEST)
        ALL_PASS = audit(RESULT)
    
    print(f'Total waktu: {elapsed_hours():.2f} jam | backbone dipakai: {list(OOF.keys())}')
else:
    print("[lewati] RUN_V7 = False")


### B.20 Audit & simpan konfigurasi ensemble v7 (v7)

In [ ]:
if RUN_V7:
    HONEST = honest_cv_scores(OOF_ALL, Y)
    FINAL_PASS = audit(RESULT, HONEST)
    
    ens_cfg = {
        'models': RESULT['names'],
        'weights': {l: dict(zip(RESULT['names'], RESULT['weights'][j].round(4).tolist())) for j, l in enumerate(LABELS)},
        'thresholds': dict(zip(LABELS, RESULT['thresholds'].tolist())),
        'oof_macro_f1': dict(zip(LABELS, RESULT['scores'].round(5).tolist())),
        'nested_cv_macro_f1': dict(zip(LABELS, HONEST.round(5).tolist())),
        'all_pillars_pass': bool(FINAL_PASS),
        'hardware': {'gpu': GPU_NAME, 'bf16': bool(USE_BF16), 'tf32': DEVICE.type == 'cuda'},
    }
    with open(os.path.join(CFG.work_dir, 'ensemble_config_v7.json'), 'w') as fh:
        json.dump(ens_cfg, fh, indent=2)
    
    if FINAL_PASS:
        print('STATUS: SEMUA 9 PILAR >= 0.93 (OOF).')
    else:
        failing = [l for j, l in enumerate(LABELS) if RESULT['scores'][j] < CFG.target_f1]
        print('STATUS: BELUM TERCAPAI. Pilar gagal:', failing)
        print('Langkah lanjutan: naikkan CFG.n_seed_variants (mis. 2), tambah epochs (5), naikkan CFG.max_len_cap ke 512,')
        print('atau tambahkan backbone lain ke CFG.extra_backbones lalu jalankan ulang notebook.')
        print('Prediksi yang sudah ada di folder preds/ akan dimuat ulang otomatis sehingga hanya backbone baru yang dilatih.')
else:
    print("[lewati] RUN_V7 = False")


In [ ]:
if RUN_V7:
    for _n in ['OOF', 'TEST', 'RESULT', 'OOF_ALL', 'TEST_ALL']:
        globals().pop(_n, None)
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
print('v7 lengkap:', preds_complete(V7_DIR, V7_REQUIRED), '| config v7:', find_v7_config())

## C. Super-Ensemble 10 Model + Stage-2 v8

Langkah-langkah (semua di CPU, tidak butuh GPU):

1. **Dekode v8** — pembersihan (HTML, zero-width, NFKC, homoglyph Kiril, URL/mention, kata bertitik/berspasi) lalu dekoder Caesar ROT-1..25 + leetspeak dengan leksikon KBBI + kamus kolokial + morfologi imbuhan, dan normalisasi istilah daerah/singkatan.
2. **Skor leksikal kontrastif** — per kata/bigram: selisih label komentar terhadap rata-rata post-nya (relevansi & per pilar) + log-odds per pilar; dihitung *out-of-fold* (GroupKFold per caption) untuk train dan dari seluruh train untuk test.
3. **Blend 10 model** — v7 (5 model, bobot per pilar dari `ensemble_config_v7.json` dinormalisasi tanpa DeBERTa) dicampur v5 (5 model, bobot tetap) dengan α per pilar ∈ {0, …, 0.30} yang memaksimalkan Macro-F1 OOF.
4. **Stage-2 v8** — statistik per post (rata-rata, kuantil-20, pilar dominan, peringkat dalam post), penyesuaian logit 7 parameter (pencarian koordinat, objektif 0,65·F1 caption + 0,35·F1 semua), dan threshold terpisah per regime (pilar dominan post / pilar sekunder / pilar minor / tanpa caption); mode 3- atau 4-regime dipilih via F1 nested-threshold.


### C.1 Load data & dekode v8

In [ ]:
import os, re, csv, json, math, html, time, glob, bisect, unicodedata, collections
import numpy as np
import pandas as pd

t0_start = time.time()
PILLARS = ['Infrastruktur', 'Kesehatan', 'Administrasi_Perizinan', 'Lingkungan_Kebersihan', 'Ketertiban_Umum',
           'Transportasi_LaluLintas', 'Sosial_Bantuan', 'Keamanan_Kriminalitas', 'Pendidikan']
DATA_ROOTS = [('/content/drive/MyDrive/Action 2026', True), ('/content/data', True), ('/content', False),
              ('/kaggle/input', True), ('./data', True), ('../data', True), ('.', False), ('..', False)]


def find_data(name, required=True):
    for root, rec in DATA_ROOTS:
        if not os.path.isdir(root):
            continue
        if os.path.isfile(os.path.join(root, name)):
            return os.path.abspath(os.path.join(root, name))
        if rec:
            hits = glob.glob(os.path.join(root, '**', name), recursive=True)
            if hits:
                return os.path.abspath(sorted(hits, key=len)[0])
    if required:
        raise FileNotFoundError(name)
    return None


def read_rows(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))


train_rows, test_rows = read_rows(find_data('train.csv')), read_rows(find_data('test.csv'))
N, N_TEST = len(train_rows), len(test_rows)


def parse_label(val):
    return 1 if str(val).strip().lower() in ('1', '1.0', 'relevan', 'true') else 0


y_true = [[parse_label(train_rows[i][p]) for p in PILLARS] for i in range(N)]
true_rel = [1 if sum(r) > 0 else 0 for r in y_true]

URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
WS_RE = re.compile(r'\s+')
TOKEN_RE = re.compile(r'[a-z0-9]+')
SPACED_WORD_RE = re.compile(r'(?<![a-zA-Z])([a-zA-Z])(?:\s+([a-zA-Z])){3,}(?![a-zA-Z])')
REPEAT3_RE = re.compile(r'(.)\1{2,}')

REGIONAL_MAP = {
    'banjer': 'banjir', 'sakola': 'sekolah', 'sakolah': 'sekolah', 'skolah': 'sekolah', 'sklh': 'sekolah',
    'alancur': 'rusak', 'rarajet': 'rusak', 'ruksak': 'rusak', 'ancur': 'rusak', 'hancur': 'rusak',
    'dalan': 'jalan', 'jln': 'jalan', 'jl': 'jalan', 'banyu': 'air', 'eweuh': 'tidak', 'euweuh': 'tidak',
    'teu': 'tidak', 'ora': 'tidak', 'kagak': 'tidak', 'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak',
    'tdk': 'tidak', 'enggak': 'tidak', 'ndak': 'tidak', 'tak': 'tidak',
    'kadanguen': 'terdengar', 'torek': 'tuli', 'pamingpin': 'pemimpin', 'janten': 'jadi',
    'etok': 'pura', 'kerjo': 'kerja', 'gawe': 'kerja', 'isok': 'bisa', 'iso': 'bisa',
    'drng': 'belum', 'durung': 'belum', 'blm': 'belum', 'sdh': 'sudah', 'udh': 'sudah', 'udah': 'sudah',
    'utk': 'untuk', 'dgn': 'dengan', 'krn': 'karena', 'kerna': 'karena', 'klw': 'kalau', 'klo': 'kalau',
    'lg': 'lagi', 'jg': 'juga', 'oge': 'juga', 'yg': 'yang', 'dlm': 'dalam',
    'bgt': 'banget', 'pisan': 'banget', 'bener': 'benar', 'bnr': 'benar', 'bntuan': 'bantuan',
    'keshtan': 'kesehatan', 'mbg': 'makan bergizi gratis', 'mb9': 'makan bergizi gratis',
    'kopdes': 'koperasi desa', 'mpp': 'mal pelayanan publik', 'fasum': 'fasilitas umum',
    'pkl': 'pedagang kaki lima', 'stnk': 'surat kendaraan pajak', 'bpjs': 'jaminan kesehatan rujukan',
    'kis': 'kartu indonesia sehat', 'kip': 'kartu indonesia pintar', 'pip': 'program indonesia pintar',
    'dtks': 'data terpadu kesejahteraan sosial', 'pkh': 'program keluarga harapan',
}

SEED_LEXICON = {
    'ktp', 'kk', 'sim', 'stnk', 'bpkb', 'npwp', 'skck', 'pdam', 'pln', 'bpjs', 'kis', 'kip', 'pkh', 'bansos', 'blt',
    'pungli', 'ppdb', 'sekolah', 'smkn', 'sman', 'smpn', 'sdn', 'puskesmas', 'rsud', 'satpol', 'dishub', 'polres',
    'polsek', 'polisi', 'damkar', 'tps', 'tpa', 'dukcapil', 'kelurahan', 'kecamatan', 'pemda', 'pemkot', 'pemkab',
    'kopdes', 'liputan', 'pabrik', 'ketintang', 'penyusup', 'dicek', 'kondisi', 'turunkan', 'terjangkau', 'fasum',
    'begal', 'parkir', 'trotoar', 'sampah', 'limbah', 'banjir', 'macet', 'korupsi', 'posyandu', 'dokter', 'pasien',
    'stunting', 'beasiswa', 'kurikulum', 'zonasi', 'tilang', 'angkot', 'halte', 'stasiun', 'kereta', 'jembatan',
    'saluran', 'drainase', 'tanggul', 'irigasi', 'asap', 'polusi', 'tawuran', 'preman', 'palak', 'narkoba', 'judol',
    'pinjol', 'penipuan', 'dtks', 'desil', 'pencairan', 'rujukan', 'pelayanan', 'perizinan', 'birokrasi', 'pajak',
    'sertifikat', 'domisili', 'simnya', 'pembukaan', 'pasokan', 'menembak', 'pemberitahuan', 'sejumlah', 'jakarta',
    'ribu', 'atau', 'cuba', 'coba', 'lihat', 'malaysia', 'liwet', 'udah', 'dasar', 'waspada', 'doang', 'anda',
    'sidang', 'paripurna', 'seumur', 'walker', 'ilegal', 'bangunan', 'konstruksi', 'renovasi', 'rehab', 'kontraktor',
    'proyek', 'plafon', 'atap', 'roboh', 'ambruk', 'rubuh', 'rusak', 'hancur', 'berlubang', 'aspal', 'beton',
}

LEXICON = set(SEED_LEXICON)
for fn, cols in [('kbbi_v6.1.0_full.csv', ('kata', 'lema')), ('colloquial-indonesian-lexicon.csv', ('slang', 'formal'))]:
    p = find_data(fn, required=False)
    if p:
        with open(p, 'r', encoding='utf-8', errors='ignore') as f:
            for r in csv.DictReader(f):
                for col in cols:
                    val = r.get(col, '')
                    if val:
                        LEXICON.update(t for t in re.findall(r'[a-z]+', val.lower()) if len(t) >= 2)
print(f'Leksikon: {len(LEXICON):,} kata')


def pre_clean_v8(s):
    if not isinstance(s, str): return ''
    s = html.unescape(s)
    s = re.sub(r'[​‌‍﻿⁠]', '', s)
    s = unicodedata.normalize('NFKC', s).lower()
    for k, v in {'а': 'a', 'е': 'e', 'і': 'i', 'о': 'o', 'р': 'p', 'с': 'c', 'у': 'y', 'х': 'x'}.items():
        s = s.replace(k, v)
    s = URL_RE.sub(' httpurl ', s)
    s = MENTION_RE.sub(' @user ', s)
    s = re.sub(r'\b([a-z])(?:\.([a-z])){2,}\b', lambda m: m.group(0).replace('.', ''), s)
    s = SPACED_WORD_RE.sub(lambda m: re.sub(r'\s+', '', m.group(0)), s)
    return WS_RE.sub(' ', s).strip()


train_pre_teks = [pre_clean_v8(r.get('Teks', '')) for r in train_rows]
train_pre_cap = [pre_clean_v8(r.get('Judul_Caption', '')) for r in train_rows]
test_pre_teks = [pre_clean_v8(r.get('Teks', '')) for r in test_rows]
test_pre_cap = [pre_clean_v8(r.get('Judul_Caption', '')) for r in test_rows]

corpus_counter = collections.Counter()
for s in train_pre_teks + test_pre_teks:
    corpus_counter.update(TOKEN_RE.findall(s))
for s in set(train_pre_cap + test_pre_cap):
    corpus_counter.update(TOKEN_RE.findall(s))

SUFFIXES = ('nya', 'lah', 'kah', 'pun', 'ku', 'mu', 'kan', 'an', 'i')
PREFIXES = (('meng', ''), ('meng', 'k'), ('meny', 's'), ('mem', ''), ('mem', 'p'), ('men', ''), ('men', 't'),
            ('me', ''), ('peng', ''), ('peng', 'k'), ('peny', 's'), ('pem', ''), ('pem', 'p'), ('pen', ''),
            ('pen', 't'), ('per', ''), ('pe', ''), ('ber', ''), ('be', ''), ('ter', ''), ('di', ''), ('ke', ''),
            ('se', ''))
LEET = str.maketrans({'0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's', '7': 't', '8': 'b', '9': 'g'})
LEET_ALT = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '7': 't', '8': 'b', '9': 'g'})


def morph_stems(w):
    stems = {w}
    for _ in range(2):
        for st in list(stems):
            for s in SUFFIXES:
                if st.endswith(s) and len(st) - len(s) >= 3:
                    stems.add(st[:-len(s)])
    for st in list(stems):
        for p, r in PREFIXES:
            if st.startswith(p) and len(st) - len(p) >= 2:
                cand = r + st[len(p):]
                if len(cand) >= 3:
                    stems.add(cand)
    stems.discard(w)
    return stems


frequent_words = {w for w, c in corpus_counter.items() if c >= 20 and w.isalpha()}
# kekuatan target dekode: 3 = di leksikon, 2 = berimbuhan dari kata leksikon/kata sering, 1 = sering di korpus
VALID_STRENGTH = {}
for w in LEXICON:
    VALID_STRENGTH[w] = 3
for w, cnt in corpus_counter.items():
    if w in VALID_STRENGTH or not w.isalpha():
        continue
    if len(w) >= 6 and any(st in LEXICON or st in frequent_words for st in morph_stems(w)):
        VALID_STRENGTH[w] = 2
    elif cnt >= 20:
        VALID_STRENGTH[w] = 1

letters = 'abcdefghijklmnopqrstuvwxyz'
CAESAR_TABLES = [(k, str.maketrans(letters, letters[k:] + letters[:k])) for k in range(1, 26)]
dec_cache = {}


def decode_word_v8(w):
    if w in dec_cache: return dec_cache[w]
    if VALID_STRENGTH.get(w, 0) >= 2:
        dec_cache[w] = w
        return w
    w_col1 = REPEAT3_RE.sub(r'\1', w)
    if w_col1 != w and VALID_STRENGTH.get(w_col1, 0) >= 2:
        dec_cache[w] = w_col1
        return w_col1
    res = w
    cand_forms = [w] if w_col1 == w else [w, w_col1]
    if len(w_col1) >= 4 and w.isalpha():
        best, best_score = None, 0.0
        cnt_w = corpus_counter.get(w, 0)
        for form in cand_forms:
            if len(form) < 4: continue
            for _, tab in CAESAR_TABLES:
                c = form.translate(tab)
                s = VALID_STRENGTH.get(c, 0)
                if s == 0: continue
                cnt_c = corpus_counter.get(c, 0)
                if len(form) <= 4 and (s < 3 or cnt_c < 3): continue
                if s == 1 and cnt_c < 5 * max(cnt_w, 1): continue
                score = s * 10.0 + math.log1p(cnt_c)
                if score > best_score:
                    best, best_score = c, score
        if best is not None:
            if not (cnt_w >= 30 and cnt_w >= corpus_counter.get(best, 0)):
                res = best
    dec_cache[w] = res
    return res


def decode_token_v8(tok):
    if tok.isdigit(): return tok
    if tok.isalpha():
        out = decode_word_v8(tok)
    else:
        out = tok
        for table in (LEET, LEET_ALT):
            t2 = tok.translate(table)
            if t2.isalpha() and len(t2) >= 3:
                d = decode_word_v8(t2)
                if VALID_STRENGTH.get(d, 0) >= 2:
                    out = d
                    break
    return REGIONAL_MAP.get(out, out)


def decode_text_v8(s):
    return TOKEN_RE.sub(lambda m: decode_token_v8(m.group(0)), s)


train_dec_teks = [decode_text_v8(s) for s in train_pre_teks]
test_dec_teks = [decode_text_v8(s) for s in test_pre_teks]
print(f'Dekode v8 selesai: {N} train + {N_TEST} test dalam {time.time() - t0_start:.1f}s')
for i in range(3):
    print(f'  {train_pre_teks[i][:80]!r}\n  -> {train_dec_teks[i][:80]!r}')

### C.2 Skor leksikal kontrastif (out-of-fold)

In [ ]:
STOP_IGNORE = {'httpurl', 'user'}


def extract_toks(s):
    toks = [w for w in re.findall(r'[a-z]{2,}', s) if w not in STOP_IGNORE]
    bigrams = [f'{toks[k]}_{toks[k + 1]}' for k in range(len(toks) - 1)]
    return len(toks), set(toks + bigrams)


train_teks_tokens = [extract_toks(s) for s in train_dec_teks]
test_teks_tokens = [extract_toks(s) for s in test_dec_teks]

# GroupKFold 5 lipatan per caption: grup terbesar -> lipatan terkecil (baris tanpa caption = grup sendiri)
cap_to_rows = collections.defaultdict(list)
for i in range(N):
    cap_to_rows[train_pre_cap[i] if train_pre_cap[i] else f'__empty_{i}'].append(i)
fold_of_row = [0] * N
fold_sizes = [0] * 5
for g_key, ilist in sorted(cap_to_rows.items(), key=lambda x: (-len(x[1]), x[0])):
    best_f = min(range(5), key=lambda f: fold_sizes[f])
    for i in ilist:
        fold_of_row[i] = best_f
    fold_sizes[best_f] += len(ilist)


def fit_lexical_maps(idx_list):
    tr_cap_rows = collections.defaultdict(list)
    for i in idx_list:
        tr_cap_rows[train_pre_cap[i]].append(i)
    cap_rel_mean, cap_pil_mean = {}, {}
    global_rel = sum(true_rel[k] for k in idx_list) / len(idx_list)
    global_pil = [sum(y_true[k][j] for k in idx_list) / len(idx_list) for j in range(9)]
    for c, ilist in tr_cap_rows.items():
        if c == '' or len(ilist) < 3:
            cap_rel_mean[c] = global_rel
            cap_pil_mean[c] = global_pil
        else:
            cap_rel_mean[c] = sum(true_rel[k] for k in ilist) / len(ilist)
            cap_pil_mean[c] = [sum(y_true[k][j] for k in ilist) / len(ilist) for j in range(9)]

    word_cnt = collections.Counter()
    word_rel_exc = collections.defaultdict(float)
    word_pil_exc = [collections.defaultdict(float) for _ in range(9)]
    word_pil_pos = [collections.Counter() for _ in range(9)]
    tot_pos = [sum(y_true[i][j] for i in idx_list) for j in range(9)]
    tot_tr = len(idx_list)
    for i in idx_list:
        c = train_pre_cap[i]
        r_bar, y_bar = cap_rel_mean[c], cap_pil_mean[c]
        r_diff = true_rel[i] - r_bar
        for w in train_teks_tokens[i][1]:
            word_cnt[w] += 1
            word_rel_exc[w] += r_diff
            for j in range(9):
                word_pil_exc[j][w] += (y_true[i][j] - y_bar[j])
                if y_true[i][j] == 1:
                    word_pil_pos[j][w] += 1

    rel_score_map = {w: val / (word_cnt[w] + 12.0) for w, val in word_rel_exc.items() if word_cnt[w] >= 3}
    pil_exc_map = [{w: val / (word_cnt[w] + 8.0) for w, val in word_pil_exc[j].items() if word_cnt[w] >= 3} for j in range(9)]
    pil_lo_map = [{} for _ in range(9)]
    for j in range(9):
        p_base = tot_pos[j] / tot_tr
        for w, cnt in word_cnt.items():
            if cnt < 4: continue
            p_w = (word_pil_pos[j][w] + 3.0 * p_base) / (cnt + 3.0)
            lo = math.log(max(1e-4, p_w) / max(1e-4, p_base))
            if abs(lo) > 0.15:
                pil_lo_map[j][w] = lo
    return rel_score_map, pil_exc_map, pil_lo_map


def score_row_lexical(tok_set, rel_score_map, pil_exc_map, pil_lo_map):
    if not tok_set:
        return 0.0, [0.0] * 9, [0.0] * 9
    r_vals = sorted([rel_score_map[w] for w in tok_set if w in rel_score_map])
    rel_s = (sum(r_vals[:2]) + sum(r_vals[-3:])) if r_vals else 0.0
    p_exc, p_lo = [0.0] * 9, [0.0] * 9
    for j in range(9):
        em, lm = pil_exc_map[j], pil_lo_map[j]
        e_vals = sorted([em[w] for w in tok_set if w in em])
        if e_vals:
            p_exc[j] = sum(e_vals[-3:]) + 0.5 * sum(e_vals[:2])
        l_vals = sorted([lm[w] for w in tok_set if w in lm and lm[w] > 0], reverse=True)
        if l_vals:
            p_lo[j] = sum(l_vals[:3])
    return rel_s, p_exc, p_lo


t0 = time.time()
oof_lex_rel, oof_lex_pillar, oof_lex_lo = [0.0] * N, [[0.0] * 9 for _ in range(N)], [[0.0] * 9 for _ in range(N)]
for vf in range(5):
    rmap, emap, lmap = fit_lexical_maps([i for i in range(N) if fold_of_row[i] != vf])
    for i in range(N):
        if fold_of_row[i] == vf:
            oof_lex_rel[i], oof_lex_pillar[i], oof_lex_lo[i] = score_row_lexical(train_teks_tokens[i][1], rmap, emap, lmap)
rmap, emap, lmap = fit_lexical_maps(list(range(N)))
test_lex = [score_row_lexical(test_teks_tokens[i][1], rmap, emap, lmap) for i in range(N_TEST)]
test_lex_rel, test_lex_pillar, test_lex_lo = [t[0] for t in test_lex], [t[1] for t in test_lex], [t[2] for t in test_lex]
print(f'Skor leksikal selesai dalam {time.time() - t0:.1f}s')

### C.3 Blend 10 model (v7 + v5)

In [ ]:
Y = np.array(y_true)
NT = N_TEST
CAP_TR, CAP_TE = train_pre_cap, test_pre_cap
FOLD = np.array(fold_of_row)
CAP_IDX = np.array([i for i in range(N) if CAP_TR[i] != ''])
LEX = {'oof': (np.array(oof_lex_rel, np.float64), np.array(oof_lex_pillar, np.float64), np.array(oof_lex_lo, np.float64)),
       'test': (np.array(test_lex_rel, np.float64), np.array(test_lex_pillar, np.float64), np.array(test_lex_lo, np.float64))}
WC_TR = np.array([len([w for w in re.findall(r'[a-z]{2,}', s) if w not in STOP_IGNORE]) for s in train_dec_teks])
WC_TE = np.array([len([w for w in re.findall(r'[a-z]{2,}', s) if w not in STOP_IGNORE]) for s in test_dec_teks])
THR_INTS = np.arange(18, 91)
THRS = THR_INTS / 100.0

v7_cfg_path = find_v7_config()
assert v7_cfg_path, 'ensemble_config_v7.json tidak ditemukan - jalankan bagian B.'
v7_cfg = json.load(open(v7_cfg_path))
print('Konfigurasi v7:', v7_cfg_path)


def load_gen(d, skip):
    out = {}
    for fn in sorted(os.listdir(d)):
        if fn.startswith('oof_') and fn.endswith('.npy') and skip not in fn[4:-4]:
            m = fn[4:-4]
            out[m] = (np.load(os.path.join(d, fn)).astype(np.float32), np.load(os.path.join(d, f'test_{m}.npy')).astype(np.float32))
    return out


v7m = load_gen(V7_DIR, 'deberta')
v5m = load_gen(V5_DIR, 'multilingual-e5')
print('v7:', list(v7m), '\nv5:', list(v5m))

o7, t7 = np.zeros((N, 9)), np.zeros((NT, 9))
for j, p in enumerate(PILLARS):
    wd = v7_cfg['weights'][p]
    ws = sum(wd.get(m, 0.0) for m in v7m)
    for m in v7m:
        w = wd.get(m, 0.0) / ws if ws > 0 else 0.0
        if w > 0:
            o7[:, j] += w * v7m[m][0][:, j].astype(np.float64)
            t7[:, j] += w * v7m[m][1][:, j].astype(np.float64)
V5_W = {'indobert-large-p1': 0.25, 'indobert-large-p1_seed142': 0.20, 'xlm-roberta-large': 0.25,
        'xlm-roberta-large_seed142': 0.20, 'indobertweet-base-uncased': 0.10}
o5, t5 = np.zeros((N, 9)), np.zeros((NT, 9))
for m, w in V5_W.items():
    if m in v5m:
        o5 += w * v5m[m][0].astype(np.float64)
        t5 += w * v5m[m][1].astype(np.float64)


def conf_table(y, p):
    # bin floor(p*100), hitung TP/FP/FN/TN untuk setiap threshold integer 18..90
    b = np.clip(np.floor(p * 100.0 + 1e-9).astype(int), 0, 100)
    pos = np.bincount(b[y == 1], minlength=101); neg = np.bincount(b[y == 0], minlength=101)
    sp = np.cumsum(pos[::-1])[::-1]; sn = np.cumsum(neg[::-1])[::-1]
    tp = sp[THR_INTS]; fp = sn[THR_INTS]; P = pos.sum(); Nn = neg.sum()
    return np.stack([tp, fp, P - tp, Nn - fp]).astype(np.float64)


def f1c(c):
    tp, fp, fn, tn = c
    a = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12), 0.0)
    b = np.where(2 * tn + fp + fn > 0, 2 * tn / np.maximum(2 * tn + fp + fn, 1e-12), 0.0)
    return 0.5 * (a + b)


def best_alpha(y, a7, a5, idx):
    best_a, best_f = 0.0, -1.0
    for ai in range(0, 35, 5):
        a = ai / 100.0
        f = f1c(conf_table(y[idx], ((1 - a) * a7 + a * a5)[idx])).max()
        if f > best_f + 1e-6: best_f, best_a = f, a
    return best_a


ALL = np.arange(N)
Po, Pt = np.zeros((N, 9)), np.zeros((NT, 9))
for j in range(9):
    a = best_alpha(Y[:, j], o7[:, j], o5[:, j], ALL)
    Po[:, j] = (1 - a) * o7[:, j] + a * o5[:, j]
    Pt[:, j] = (1 - a) * t7[:, j] + a * t5[:, j]
    print(f'  {PILLARS[j]:25s} alpha v5 = {a:.2f} | Macro-F1 OOF blend = {f1c(conf_table(Y[:, j], Po[:, j])).max():.5f}')

### C.4 Stage-2 v8 (multi-regime per post) & submission

In [ ]:
def group_stats(P, caps):
    n = len(P); pmean = P.copy(); pq20 = P.copy(); prank = np.full((n, 9), 0.5); prim = np.full(n, -1); wmr = np.full(n, 0.5)
    gm = collections.defaultdict(list)
    for i, c in enumerate(caps): gm[c if c else f'__empty_{i}'].append(i)
    for k, il in gm.items():
        ng = len(il)
        if k.startswith('__empty_') or ng < 4: continue
        il = np.array(il); sub = P[il]; srt = np.sort(sub, 0); m = sub.mean(0)
        pmean[il] = m; pq20[il] = srt[int(0.20 * (ng - 1))]; prim[il] = int(np.argmax(m))
        mx = np.sort(sub.max(1)); wmr[il] = np.searchsorted(mx, sub.max(1), side='right') / ng
        for j in range(9): prank[il, j] = np.searchsorted(srt[:, j], sub[:, j], side='right') / ng
    return pmean, pq20, prim, wmr, prank


def regime(j, prim, pmean):
    # 2 = pilar dominan post, 1 = pilar sekunder (rata-rata post >= 0.11), 0 = minor, -1 = tanpa caption
    r = np.where(pmean[:, j] >= 0.11, 1, 0); r = np.where(prim == j, 2, r); return np.where(prim == -1, -1, r)


def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))


def adj(j, P, st, wc, lex, prm):
    a_exc, a_lo, a_sec, a_rel, a_pen, a_flr, a_rnk = prm
    pmean, pq20, prim, wmr, prank = st; rel, pil, lo = lex
    p = P[:, j]; z = logit(p); lp, ll = pil[:, j], lo[:, j]
    if a_exc: z = z + a_exc * lp
    if a_lo: z = z + a_lo * ll
    if a_sec: z = z + np.where((prim != j) & (ll >= 1.0) & (lp >= 0.10), a_sec * (lp + 0.15 * ll), 0.0)
    if a_rel: z = z + a_rel * rel
    hp = prim >= 0
    if a_flr: z = z + np.where(hp, a_flr * (p - pq20[:, j]), 0.0)
    if a_rnk: z = z + np.where(hp, a_rnk * (prank[:, j] - (1.0 - pmean[:, j])), 0.0)
    if a_pen > 0:
        c1 = hp & (wmr < 0.28) & (wc <= 6) & (ll < 1.0) & (lp <= 0.05)
        c2 = hp & ~c1 & (wc <= 2) & (ll < 0.5) & (lp <= 0.0)
        z = z - np.where(c1, a_pen, 0.0) - np.where(c2, 0.7 * a_pen, 0.0)
    return 1 / (1 + np.exp(-z))


def solve3(y, p, reg, idx):
    T = {r: conf_table(y[idx][m], p[idx][m]) for r, m in [(2, reg[idx] == 2), (1, np.isin(reg[idx], (1, 0))), (-1, reg[idx] == -1)]}
    has_n = (reg[idx] == -1).any(); k2 = k1 = kn = 32; best = -1.0
    for _ in range(3):
        s = f1c(T[2] + T[1][:, [k1]] + T[-1][:, [kn]]); c = int(np.argmax(s))
        if s[c] > best: best, k2 = s[c], c
        s = f1c(T[2][:, [k2]] + T[1][:, k2:] + T[-1][:, [kn]]); c = int(np.argmax(s))
        if s[c] > best: best, k1 = s[c], c + k2
        if has_n:
            s = f1c(T[2][:, [k2]] + T[1][:, [k1]] + T[-1]); c = int(np.argmax(s))
            if s[c] > best: best, kn = s[c], c
    k1 = max(k1, k2)
    return (THRS[k2], THRS[k1], THRS[k1], THRS[kn]), best


def solve4(y, p, reg, idx):
    T = {r: conf_table(y[idx][reg[idx] == r], p[idx][reg[idx] == r]) for r in (2, 1, 0, -1)}
    has_n = (reg[idx] == -1).any(); k2, k1, k0, kn = 25, 38, 52, 32; best = -1.0
    for _ in range(3):
        s = f1c(T[2] + T[1][:, [k1]] + T[0][:, [k0]] + T[-1][:, [kn]]); c = int(np.argmax(s))
        if s[c] > best: best, k2 = s[c], c
        k1 = max(k1, k2); best = -1.0
        s = f1c(T[2][:, [k2]] + T[1][:, k2:] + T[0][:, [k0]] + T[-1][:, [kn]]); c = int(np.argmax(s))
        if s[c] > best: best, k1 = s[c], c + k2
        k0 = max(k0, k1); best = -1.0
        s = f1c(T[2][:, [k2]] + T[1][:, [k1]] + T[0][:, k1:] + T[-1][:, [kn]]); c = int(np.argmax(s))
        if s[c] > best: best, k0 = s[c], c + k1
        if has_n:
            s = f1c(T[2][:, [k2]] + T[1][:, [k1]] + T[0][:, [k0]] + T[-1]); c = int(np.argmax(s))
            if s[c] > best: best, kn = s[c], c
    return (THRS[k2], THRS[k1], THRS[k0], THRS[kn]), best


GRIDS = [[0.0, 0.5, 1.0, 1.8, 2.6], [0.0, 0.06, 0.12, 0.20], [0.0, 0.5, 1.0, 1.8], [0.0, 0.4, 0.8, 1.4],
         [0.0, 0.3, 0.6, 1.0], [0.0, 0.4, 0.8], [0.0, 0.3, 0.6, 1.0]]


def apply_thr(p, reg, t):
    tu = np.select([reg == 2, reg == 1, reg == 0], [t[0], t[1], t[2]], t[3]); return (p >= tu).astype(int)


def _mf1(y, yp):
    tp = int(((y == 1) & (yp == 1)).sum()); fp = int(((y == 0) & (yp == 1)).sum()); fn = int(((y == 1) & (yp == 0)).sum()); tn = len(y) - tp - fp - fn
    return float(f1c(np.array([[tp], [fp], [fn], [tn]], float))[0])


def fit_mode(j, y, P, st, reg, idx, solver, cap_w=0.65):
    # pencarian koordinat 7 parameter: objektif 0.65*F1 baris ber-caption + 0.35*F1 semua baris
    prm = [0.0] * 7; best_obj = -1.0; cap_idx = idx[np.isin(idx, CAP_IDX)]
    for s_, grid in enumerate(GRIDS):
        bv = prm[s_]
        for v in grid:
            c = prm[:]; c[s_] = v; p = adj(j, P, st, WC_TR, LEX['oof'], c)
            obj = cap_w * solver(y, p, reg, cap_idx)[1] + (1 - cap_w) * solver(y, p, reg, idx)[1]
            if obj > best_obj + 1e-6: best_obj, bv = obj, v
        prm[s_] = bv
    p = adj(j, P, st, WC_TR, LEX['oof'], prm); thr, f_all = solver(y, p, reg, idx)
    # F1 nested-threshold (threshold di-fit pada 4 lipatan, dievaluasi pada lipatan ke-5) untuk memilih 3Reg vs 4Reg
    yp = np.zeros(N, int)
    for vf in np.unique(FOLD[idx]):
        tri = idx[FOLD[idx] != vf]; vai = idx[FOLD[idx] == vf]; t_, _ = solver(y, p, reg, tri); yp[vai] = apply_thr(p[vai], reg[vai], t_)
    return prm, thr, f_all, _mf1(y[idx], yp[idx])


def fit_label(j, y, P, st, idx):
    reg = regime(j, st[2], st[0])
    r3 = fit_mode(j, y, P, st, reg, idx, solve3); r4 = fit_mode(j, y, P, st, reg, idx, solve4)
    return ('4Reg', r4) if r4[3] > r3[3] + 1e-5 else ('3Reg', r3)


test_ids = [r['ID'] for r in test_rows]
st_tr, st_te = group_stats(Po, CAP_TR), group_stats(Pt, CAP_TE)
sub = pd.DataFrame({'ID': [int(x) if x.strip().isdigit() else x for x in test_ids]})
rows = []
for j in range(9):
    mode, (prm, thr, f_all, nest) = fit_label(j, Y[:, j], Po, st_tr, ALL)
    pt = adj(j, Pt, st_te, WC_TE, LEX['test'], prm)
    sub[PILLARS[j]] = apply_thr(pt, regime(j, st_te[2], st_te[0]), thr)
    rows.append(dict(pilar=PILLARS[j], mode=mode, oof_F1=round(f_all, 5), nested_thr_F1=round(nest, 5),
                     thr_dominan=thr[0], thr_sekunder=thr[1], thr_minor=thr[2], thr_tanpa_caption=thr[3], params=prm))
rep = pd.DataFrame(rows)
print(rep.to_string(index=False))
print(f'Mean Macro-F1 OOF = {rep.oof_F1.mean():.5f} | nested-threshold = {rep.nested_thr_F1.mean():.5f}')

assert len(sub) == N_TEST and sub.shape[1] == 10 and sub.isna().sum().sum() == 0
SUB_PATH = os.path.join(WORK_DIR, 'submission.csv')
sub.to_csv(SUB_PATH, index=False)
sub.to_csv(os.path.join(WORK_DIR, 'submission_v8_super10.csv'), index=False)
print('Tersimpan:', SUB_PATH, '| baris:', len(sub) + 1, '(header + prediksi)')
print(sub[PILLARS].mean().round(4))
print(f'Total bagian C: {time.time() - t0_start:.0f}s')